**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/kaku-2025"], check=True)
    os.chdir("/content/encore/entries/kaku-2025")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Kaku, Liu et al. (2025) from dandiset 001619

Paper. Kaku H, Liu LD, Gao R, West S, Liao S-M, Finkelstein A, Kleinfeld D, Thomas A, Tipparaju SL,
Svoboda K, Li N (2025). A brainstem map of orofacial rhythms. bioRxiv,
[doi:10.1101/2025.01.27.635041](https://doi.org/10.1101/2025.01.27.635041). The claims file was written
from the preprint, version 1 (January 27, 2025), its Methods and the Extended Data legends, and from the
paper's analysis code (github.com/NuoLiLabBCM/KakuEtAl2025 at commit fbc0abe).

Dandiset. [DANDI:001619](https://dandiarchive.org/dandiset/001619/0.251027.1953), version
0.251027.1953. The paper cites no version (its data statement says the data will be deposited upon
publication; the dandiset was created later). This is the only published version, and it is the
most recent; the draft has the same 335 assets and size. It holds the brainstem Neuropixels recordings
of the multi-directional licking task: one NWB file per probe insertion, with units, quality metrics,
CCF positions, DeepLabCut tracking and the airflow (breathing) trace, per trial.

What is not in it. The cued licking task with ALM, latSC and brainstem photostimulation, the rat
recordings, the raw videos, and the premotor tracing data are not in the dandiset.

Outside input. Region names in the files are Allen CCF names; their acronyms and parent regions
come from the Allen structure tree, read from the iblatlas repository at a pinned commit
(`kaku.ONTOLOGY_URL`). It maps names only. The regions the CCF does not label (preBotzinger complex,
parts of IRN, the areas next to AMB and NTS) are built from the Allen CCFv3 2017 annotation volume at 25
um (`kaku.CCF_ANNOTATION_URL`).

Revision after round 1 of verification. Lick frequencies within bouts take only pairs inside one bout;
the regions above are built from the annotation volume, not from labelled units; the 3D tracks are read
at each session's trial offset; the peak-difference claims carry a random-split null; Fig 4e units use
the 40 fastest and 40 slowest cycles as the claims file says, with the code's selection beside them;
and ed1, ed1bc, fig1l, fig4d, ed7 and ed8i show the outputs the verifier asked for.

Revision after round 2. No outcome is changed. Three changes are proposed for a third round, with the
analysis below: units-count to not reproduced (the criterion defines no partial outcome), and fig2f and
ed8bc to blocked, methods_underspecified (the two readings of AMB's extent give different outcomes).
The data check compares the airflow lag of subjects DL029 to DL034 with the others; fig1f and fig1g
give results per subject with those subjects apart; fig2f and ed8bc give the mice behind their units;
and sentences the round-2 report found unsupported are corrected.

Revision after round 3. The verifier agreed with the three proposed changes, and they are applied:
units-count is not reproduced, and fig2f and ed8bc are blocked, methods_underspecified.

| Claim | Tested here |
|---|---|
| code-dataset-link, units-count, region-spike-rates | yes |
| fig1d, fig1f (two), fig1g, fig1i, ed1, ed1bc, fig1k, fig1l (two), fig1n (two) | yes |
| fig2 tuned counts, swallow counts, fig2f, fig2g, fig2h, ed6e, ed8bc, ed8e, ed4c | yes (fig2f, fig2h and ed8bc blocked, methods_underspecified) |
| fig4e, fig4f | yes |
| fig4d, fig4g, fig4g lags, fig4h | run in part; blocked, data_missing (recordings not deposited) |
| ed8i (lick direction) | part without the sign run; blocked, metadata_insufficient |
| ed7 (GLM) | cost measured; blocked, too_large |
| ed7g, ed7h (CAE) | blocked, data_missing (raw videos) |
| fig3a, fig3b (premotor atlas) | blocked, other (public source found, not analysed in this run) |
| ed12c, ed2 (rats), fig3ce, fig3dfh, ed10, fig4b, fig4c, ed11b, fig5b, fig5c, fig5e, fig5f | blocked, data_missing |

The notebook streams every file once (50 to 71 minutes in the runs made so far, most of it reading
the 25 kHz airflow and fitting tuning curves; the last cell prints the run time) and computes each claim from what it reads.

In [ ]:
import time
import warnings

import numpy as np
import pandas as pd
from scipy import stats

import kaku

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
t_start = time.time()

assets = kaku.list_assets()
files, units = kaku.survey(assets)
onto = kaku.ontology()
regions = kaku.region_table(units.region.dropna().unique(), onto)
print(f"{len(assets)} files, {assets['size'].sum() / 1e9:.1f} GB, read in {time.time() - t_start:.0f} s (metadata)")

In [ ]:
jobs = []
for i, ((sub, ses), g) in enumerate(assets.groupby(["subject", "session"], sort=True)):
    jobs.append(((sub, ses), g.url.iloc[0], dict(zip(g.probe, g.url)), 1000 + i))
t0 = time.time()
res = kaku.run_sessions(jobs, workers=5)
print(f"{len(res)} sessions processed in {(time.time() - t0) / 60:.1f} min")

In [ ]:
# Unit table: every unit of the survey, with the per-session results joined to the code's good units.
units["pos"] = units.groupby(["subject", "session", "probe"]).cumcount()
units = units.merge(regions.rename(columns={"name": "region"}), on="region", how="left")
units["ancestors"] = units["ancestors"].apply(lambda a: a if isinstance(a, tuple) else ())
units["good"] = kaku.good_units(units)
units["good_drift"] = kaku.good_units(units, drift=True)
rows = []
for r in res:
    sub, ses = r["key"]
    for probe, us in r["units"].items():
        for u in us:
            rows.append({"subject": sub, "session": ses, "probe": probe, "pos": u["unit_index"],
                         **{k: v for k, v in u.items() if not k.endswith(("_cnt_4", "_cnt_2", "_cnt_5", "_occ_4", "_occ_2", "_occ_5", "_cen_4", "_cen_2", "_cen_5")) and k != "lick_counts"}})
T = pd.DataFrame(rows).merge(units, on=["subject", "session", "probe", "pos"], how="left")
T["hemi"] = np.where(T.ccf_x < 5700, "left", "right")
def under(d):
    return T.ancestors.apply(lambda a: d in a).to_numpy()
for d in ("CB", "P", "MY", "VNC", "IRN", "PARN", "XII", "V", "AMB", "NTS", "IC", "NLL"):
    T["in_" + d] = under(d)
print("Good units with session results:", len(T), "of", int(units.good.sum()))

## Data check

In [ ]:
fs_ = files.copy()
print("Subjects:", fs_.subject.nunique(), "; sessions:", fs_.groupby(["subject", "session"]).ngroups, "; files (insertions):", len(fs_))
print("Sex per subject:", fs_.groupby("subject").sex.first().value_counts().to_dict())
print("Age field values (first 8):", sorted(fs_.age.unique())[:8], "...")
print("Genotype field:", fs_.genotype.unique().tolist())
print("Stimulus field:", fs_.stimulus.unique().tolist(), "; acquisition groups:", sorted({a for x in fs_.acquisition for a in x}),
      "; stimulus/presentation:", sorted({a for x in fs_.stimulus_presentation for a in x}))
print("Trial table columns:", sorted({c for x in fs_.trial_columns for c in x}), "; largest start_time over all files:", fs_.trial_start_max.max())
print("Trial duration range (s):", fs_.trial_duration_min.min(), "to", fs_.trial_duration_max.max())
print("Trials per file:", fs_.n_trials.describe()[["min", "50%", "max"]].to_dict())
print("Files per session:", fs_.groupby(["subject", "session"]).size().value_counts().to_dict())
print("Probe insertions per subject:")
print(fs_.groupby("subject").size().to_string())
print("Tracking info (video Hz, frames per trial, seconds) values:", fs_.tracking_info.unique().tolist())
print("Probe types:", fs_.ins_probe_type.unique().tolist(), "; insertion targets:", fs_.ins_recordable_brain_regions.value_counts().to_dict())

In [ ]:
# Formats: spike index, table lengths, DLC rows, airflow samples
print("spike_times_index read as start offsets matches the per-trial spike counts in", int(fs_.index_is_start.sum()), "of", len(fs_), "files;",
      "read as end offsets (the NWB convention) in", int(fs_.index_is_end.sum()))
L = fs_[[c for c in fs_.columns if c.startswith("len_")]]
odd = fs_[L.nunique(axis=1) > 1]
print("Files whose units tables differ in length:", len(odd))
print(odd[["subject", "session", "probe", "len_metrics", "len_units_id", "len_unit_info", "len_pairs", "len_histology"]].to_string(index=False))
print("DLC rows = trials x 1471 in", int((fs_.n_dlc_rows_side == fs_.n_trials * 1471).sum()), "files (side),",
      int((fs_.n_dlc_rows_bottom == fs_.n_trials * 1471).sum()), "(bottom); 3D rows = (trials - 1) x 1471 in",
      int((fs_.n_3d_rows == (fs_.n_trials - 1) * 1471).sum()), "files")
print("Airflow samples equal the sum of data_idx in", int((fs_.n_breath == fs_.breath_idx_sum).sum()), "files")
two = fs_.groupby(["subject", "session"]).filter(lambda g: len(g) == 2)
same = two.groupby(["subject", "session"]).agg(b=("breath_head_hash", "nunique"), j=("jaw_head_hash", "nunique"), n=("n_breath", "nunique"))
print("Two-probe sessions:", len(same), "; with identical airflow (first 200,000 samples), identical side jaw tracking (first 20,000 rows) and equal airflow length:",
      int(((same.b == 1) & (same.j == 1) & (same.n == 1)).sum()))
rs = pd.DataFrame([{"subject": r["key"][0], "session": r["key"][1], "trials": r["n_trials_total"], "good_trials": r["n_good_trials"],
                    "video_fs": r["video_fs"], "trial_len": r["trial_len"], "breath_dt": r["breath_dt"], "breath_trials": r["n_breath_trials"],
                    "licks": len(r["mt"]["tongue_onset"]), "inspirations": len(r["mt"]["inspir_onset"]), "bouts": len(r["mt"]["bout_on"]),
                    "swallows": len(r["swallow_times"])} for r in res])
print("\nPer session (173):")
print(rs.describe().round(4).to_string())
print("Airflow sampling interval (s):", rs.breath_dt.unique().tolist(), "(Methods: 25 kHz)")

Every file stores each unit's spikes in `units/spike_times` with `spike_times_index` holding the
offset where the unit starts. In NWB the index holds where each unit ends, so a standard reader (pynwb)
gives each unit the spikes of the unit before it. The per-unit, per-trial spike counts (`unit and
trial pairs`) agree with the start-offset reading in every file, and spikes are read that way here.
In the files listed above the histology table (region and CCF position) has more rows than the units,
so its rows cannot be matched to units with certainty; like the paper's code, the rows are taken by
position. Spike times are relative to the trial start; every trial starts at time 0 in the trial
table, so the trials cannot be placed on a session clock. The behaviour of the two probes of a session
is the same recording deposited twice: it is read once per session. The age field (for example
`P273W`, 273 weeks) is not plausible for these mice. The stimulus field says `photostim`, but the
files hold no photostimulation or cue times.

In [ ]:
gu = units[units.good]
print("Unit metrics of the code's good units (ranges):")
print(gu[["unit_amp", "avg_firing_rate", "isi_violation", "presence_ratio", "amplitude_cutoff", "drift_metric"]].describe().round(3).to_string())
print("\nCCF coordinates of good units (um): x", gu.ccf_x.min(), "to", gu.ccf_x.max(), "; y", gu.ccf_y.min(), "to", gu.ccf_y.max(), "; z", gu.ccf_z.min(), "to", gu.ccf_z.max())
print("Region names not found in the Allen structure tree:", regions[regions.acronym.isna()].name.tolist())
lab = units.dropna(subset=["ccf_x"])
print("Units whose 'left'/'right' unit_location agrees with ccf_x on the side of 5700 um:",
      round(float(np.mean((lab.info_unit_location.str.startswith("left")) == (lab.ccf_x < 5700))), 3))
# axes: which CCF coordinate runs anterior-posterior? cerebellum units should be dorsal (small y) and medulla ventral
print("Median ccf_y (DV) of CB units:", T[T.in_CB].ccf_y.median(), "; of MY units:", T[T.in_MY].ccf_y.median())
print("Median ccf_z (AP) of P units:", T[T.in_P].ccf_z.median(), "; of MY units:", T[T.in_MY].ccf_z.median())

`ccf_x` is medial-lateral, `ccf_y` dorsal-ventral (cerebellum above medulla) and `ccf_z`
anterior-posterior (pons anterior of medulla), in um, as the paper's code plots them; the files do not
name the axes. The midline of the CCF is at 5700 um, but the `unit_location` text ("left Medulla",
"right Medulla") puts almost every unit on the side opposite to the one that reading of `ccf_x` gives
if it increases to the right. Which side is left is therefore not settled by the files; no claim tested
here depends on it.

In [ ]:
# Clock check. Spikes, tracking and airflow are stored per trial on separate arrays. If they share the
# trial clock, phase tuning to jaw and breathing should fall when each trial's spikes are paired with the
# next trial's phases.
for mod, col in (("jaw", "jaw_mi_grid_4"), ("breathing", "br_mi_grid_4")):
    shift = "jaw_mi_shift" if mod == "jaw" else "br_mi_shift"
    top = T[T[col] > 0.6]
    print(f"{mod}: units with grid MI > 0.6, true pairing {len(top)}; their median MI {top[col].median():.3f} true, "
          f"{top[shift].median():.3f} with the next trial's phases; units above 0.6 after the shift: {(T[shift] > 0.6).sum()}")
by = T.groupby("subject").apply(lambda g: pd.Series({"n_true": (g.jaw_mi_grid_4 > 0.6).sum(), "n_shift": (g.jaw_mi_shift > 0.6).sum(),
                                                     "br_true": (g.br_mi_grid_4 > 0.6).sum(), "br_shift": (g.br_mi_shift > 0.6).sum()}))
print("\nPer subject, units with MI > 0.6, true and shifted pairing:")
print(by.T.to_string())
# Behaviour: inspirations and tongue onsets lie on the same per-trial clock; the lick-breath relation is tested in fig1f.

In [ ]:
# Airflow against tongue and jaw (verifier, round 1). Per session, the cross-correlation of the z-scored
# airflow with the tongue-visible indicator and with jaw_y, lags -0.5 to 0.5 s (positive lag: airflow
# later); per subject the mean curve, its largest |r| and the lag of it, and the spread of the session lags.
lags = np.arange(-147, 148) * kaku.FRAME_S
XC = pd.DataFrame([{"subject": r["key"][0], "signal": name, "c": r["xcorr"][name]} for r in res for name in ("tongue", "jaw")])
out = []
for (sub, name), g in XC.groupby(["subject", "signal"]):
    m = np.mean(np.vstack(g.c), 0)
    k = np.argmax(np.abs(m))
    sl = np.array([lags[np.argmax(np.abs(c))] for c in g.c]) * 1000
    out.append({"subject": sub, "signal": name, "sessions": len(g), "r at extreme": m[k], "lag of extreme (ms)": lags[k] * 1000,
                "session lags 25% (ms)": np.percentile(sl, 25), "session lags 75% (ms)": np.percentile(sl, 75)})
XCS = pd.DataFrame(out)
print(XCS.round(3).to_string(index=False))
for name in ("tongue", "jaw"):
    d = XCS[XCS.signal == name]
    print(f"{name}: lag of the subject-mean extreme, median {d['lag of extreme (ms)'].median():.1f} ms, range {d['lag of extreme (ms)'].min():.1f} "
          f"to {d['lag of extreme (ms)'].max():.1f} ms; r at the extreme from {d['r at extreme'].min():.3f} to {d['r at extreme'].max():.3f}, "
          f"positive in {int((d['r at extreme'] > 0).sum())} of {len(d)} subjects")
print("Lag range of the plotted curves: -500 to 500 ms; all points shown.")
# Subjects with few inspirations inside a protrusion (fig1f) against the others (verifier, round 2): lags of the
# subject-mean curves, over the subjects whose tongue curve peaks within the first breath cycle (< 200 ms).
LOW = ["DL029", "DL030", "DL031", "DL032", "DL033", "DL034"]
first = XCS[(XCS.signal == "tongue") & (XCS["lag of extreme (ms)"] < 200)].subject
for name in ("tongue", "jaw"):
    d = XCS[(XCS.signal == name) & XCS.subject.isin(first)].set_index("subject")["lag of extreme (ms)"]
    lo, hi = d[d.index.isin(LOW)], d[~d.index.isin(LOW)]
    print(f"{name}: DL029 to DL034 ({len(lo)}) lags {lo.min():.1f} to {lo.max():.1f} ms, median {lo.median():.1f}; other subjects ({len(hi)}) "
          f"{hi.min():.1f} to {hi.max():.1f} ms, median {hi.median():.1f}; DL029 to DL034 above every other subject: {int((lo > hi.max()).sum())} of {len(lo)}; "
          f"Mann-Whitney p = {stats.mannwhitneyu(lo, hi).pvalue:.2g}")

# 3D tracking `idx` against the DLC `trial` field (verifier, round 1)
IDX = pd.DataFrame([{"subject": r["key"][0], "session": r["key"][1], **{f"r, offset {o}": v for o, v in r["idx3d_corr"].items()},
                     "offset used": r["idx3d_offset"]} for r in res])
print("\n3D idx = DLC trial + offset; sessions by the offset with the highest correlation:", IDX["offset used"].value_counts().to_dict())
print(IDX[[c for c in IDX.columns if c.startswith("r,")]].describe().round(3).to_string())
print("Subjects by offset used:", IDX.groupby("offset used").subject.apply(lambda s: sorted(set(s))).to_dict())
best = IDX[[c for c in IDX.columns if c.startswith("r,")]].max(1)
print(f"Correlation at the offset used: median {best.median():.3f}, minimum {best.min():.3f}")

Jaw phase tuning is present with the true pairing (3,476 units above MI 0.6, median 0.80) and falls
when spikes are paired with the next trial's tracking (median 0.13, 308 units above 0.6), in every
subject. Breathing tuning falls likewise overall (950 units, median 0.74, against 0.16 and 313 units),
but in four subjects (DL004, DL005, DL010, DL020) the shifted pairing gives as many units above 0.6 as
the true one (5 against 7, 7 against 9, 14 against 14, 3 against 6), so for their airflow the shared
clock is not established by this check. This concerns alignment within trials; the trials cannot be
placed on one session clock, since every trial starts at 0.

Airflow against tongue and jaw (added in round 1 of verification). The cross-correlation of the airflow
with the tongue-visible indicator peaks at a positive lag in every subject: median 85 ms over subjects,
54 to 112 ms in 28 of 31, and about 250 ms (the next breath cycle) in DL007, DL010 and DL027. The
correlations are weak (r 0.035 to 0.106), so this check is consistent with one clock within trials but
cannot exclude offsets of a few tens of milliseconds. Subjects DL029 to DL034, where fewer inspirations
fall inside a protrusion (fig1f), peak later: tongue lags 88.4 to 112.2 ms (median 103.7) against 54.4 to
95.2 ms (median 78.2) in the 22 other subjects that peak within the first cycle, with 5 of the 6 above
all of them (Mann-Whitney p = 0.00054); jaw lags 91.8 to 108.8 ms (median 100.3) against a median of
85.0 (p = 0.0022). So in these six subjects the airflow may lag the video by a further 20 to 30 ms; fig1f
shows the effect of removing that difference.

The 3D tracking. In 154 of 173 sessions the 3D `idx` field matches the DLC `trial` field (median
correlation 0.98 between the 2D and 3D tongue tracks at offset 0); in 19 sessions, in 15 subjects, it
matches trial - 1. Each session's 3D tracks are read at its best offset (`kaku.read_session`); round 1
read every file at offset 0. This bears on ed8e and on the lick tips of ed8i.

## code-dataset-link

> The dandiset is the data the paper's analysis code reads, so that the code's choices fixed in this
> file apply to it.

In [ ]:
need = {"spike times per trial": "units/spike_times with unit and trial pairs",
        "unit quality metrics": "analysis/unit_metrics (presence_ratio, amplitude_cutoff, avg_firing_rate, isi_violation, unit_amp, drift_metric)",
        "CCF coordinates": "units/histology (ccf_x, ccf_y, ccf_z, unit_region)",
        "airflow at 25 kHz with timestamps": "breathing data, timestamps and data_idx",
        "side and bottom DLC jaw and tongue with likelihoods": "DLC_tracking side- and bottom-facing camera jaw, tongue, lickport",
        "3D jaw and tongue tracks": "3D_tracking jaw_side, tongue_bottom, lickport_bottom"}
for k, v in need.items():
    print(f"{k:55s} <- {v}")
n_code = int(units.good.sum()); n_drift = int(units.good_drift.sum())
print(f"\nUnits passing get_good_units over all files: {n_code} ({n_code / 20300 - 1:+.1%} from 20,300); with the drift metric: {n_drift} ({n_drift / 20300 - 1:+.1%})")
print("Sessions processed through the code's steps without error:", len(res), "of", len(jobs))

**Result: reproduced.** Every input of process_mat.py can be built from the NWB files (listed above), and
the code's unit filter keeps 20,109 units, 0.9% below 20,300. With the Methods' drift metric added it
keeps 16,957 (16.5% below). The spike index needs the reading described in the data check.

## units-count

> Across 339 insertions in 31 mice, 20,300 single units passed quality control (60 +- 22 per probe),
> 5,700 in the cerebellum, 3,700 in the pons and 10,900 in the medulla.

In [ ]:
def counts(mask):
    g = units[mask]
    per = g.groupby(["subject", "session", "probe"]).size().reindex(pd.MultiIndex.from_frame(files[["subject", "session", "probe"]]), fill_value=0)
    anc = g.ancestors
    cb = anc.apply(lambda a: "CB" in a)
    arb = g.acronym.isin(["arb", "cbc"])
    return {"units": len(g), "per probe mean": per.mean(), "per probe sd": per.std(),
            "CB": int(cb.sum()), "CB + arbor vitae": int((cb | arb).sum()),
            "P": int(anc.apply(lambda a: "P" in a).sum()), "MY": int(anc.apply(lambda a: "MY" in a).sum()),
            "other": int((~anc.apply(lambda a: any(d in a for d in ("CB", "P", "MY"))) & ~arb).sum())}
tab = pd.DataFrame({"code filter": counts(units.good), "with drift metric": counts(units.good_drift)}).T
print(tab.round(1).to_string())
print("\nPaper: 20,300 units (tolerance 2,030), 60 +- 22 per probe, CB 5,700 (570), P 3,700 (370), MY 10,900 (1,090); 339 insertions, 31 mice")
print("Dandiset: insertions (files)", len(files), "; mice", files.subject.nunique())
print("Units with amplitude > 150 uV (all other metrics ignored):", int((units.unit_amp > 150).sum()), "(Extended Data Fig 4c: 19,000)")
print("\nGood units outside CB, P and MY, by region (top 15):")
g = units[units.good]
out = g[~g.ancestors.apply(lambda a: any(d in a for d in ("CB", "P", "MY")))]
print(out.region.value_counts().head(15).to_string())
print("\nPer subject (code filter):")
print(units[units.good].groupby("subject").size().to_string())

**Result: not reproduced.** The total (20,109 by the code's filter) and the mean per probe (60.0)
are within tolerance, and there are 31 mice, but the dandiset holds 335 insertions, not 339, and the
pons (3,021) and medulla (8,992) counts are below the paper's. The cerebellum holds 4,157 units, or
5,385 with the arbor vitae (within tolerance of 5,700). 2,711 good units lie outside these regions
(midbrain, fibre tracts and others). The spread per probe is 32.9 where the paper gives 22. The
criterion defines no partial outcome, and the insertions (an exact count), the pons and the medulla
are outside it, so the claim is not reproduced, as for region-spike-rates and fig1f-licking-mode-switch.
How the paper groups fibre-tract units does not change this, since the insertion count fails. This was
partially reproduced until round 3 of verification, which agreed with the change.

## region-spike-rates

> Neurons in the cerebellum (22 +- 29 spikes/s) and vestibular nucleus (38 +- 32) fired faster than
> neurons in the medulla (11 +- 14).

In [ ]:
rows = []
for rate_name, col in (("(a) avg_firing_rate", "avg_firing_rate"), ("(b) spikes in the first 5 s of good trials", "rate_5s")):
    for med_name, med in (("(i) MY with VNC", T.in_MY), ("(ii) MY without VNC", T.in_MY & ~T.in_VNC)):
        r = {"rate": rate_name, "medulla": med_name}
        for name, m in (("CB", T.in_CB), ("VNC", T.in_VNC), ("MY", med)):
            v = T.loc[m, col]
            r[name + " mean"], r[name + " sd"], r[name + " n"] = v.mean(), v.std(), int(m.sum())
        rows.append(r)
rr = pd.DataFrame(rows)
print(rr.round(2).to_string(index=False))
print("\nPaper: CB 22 (+-2.2), VNC 38 (+-3.8), MY 11 (+-1.1) spikes/s")
rr["all within"] = (abs(rr["CB mean"] - 22) <= 2.2) & (abs(rr["VNC mean"] - 38) <= 3.8) & (abs(rr["MY mean"] - 11) <= 1.1)
print(rr[["rate", "medulla", "all within"]].to_string(index=False))
print("\nCB mean rate with the arbor vitae included, reading (a):", round(T.loc[T.in_CB | T.acronym.isin(["arb", "cbc"]), "avg_firing_rate"].mean(), 2))

**Result: not reproduced.** Cerebellum and medulla match: 21.3 spikes/s in the cerebellum and
10.9 in the medulla without the vestibular nuclei (reading (a); 21.4 and 11.3 under (b)), against 22 and
11. The vestibular nuclei fire at 27.2 spikes/s under both rate readings, outside 38 +- 3.8, though
still faster than the medulla. With the vestibular nuclei counted in the medulla (i), the medulla mean
is 13.2 to 13.6, outside tolerance. The criterion asks for all three means under one reading and
defines no partial outcome.

## Behaviour: licking, breathing and swallowing (Figure 1, Extended Data Figure 1)

Tongue onsets, lick bouts, inspiration onsets and swallows follow the paper's code (helper docstring).
Times are on the code's timeline: the first 1471 frames of each good trial placed end to end. The two
probes of a session share one behaviour recording, so behaviour is counted once per session.

In [ ]:
sess = pd.DataFrame([{"subject": r["key"][0], "session": r["key"][1]} for r in res])
def trial_of(t):
    return np.floor(t / kaku.TRIAL_S).astype(int)
def bout_of(t, mt):
    """Index of the code's lick bout holding each time, -1 outside bouts."""
    on, off = mt["bout_on"], mt["bout_off"]
    i = np.searchsorted(on, t, side="right") - 1
    ok = i >= 0
    ok[ok] = t[ok] <= off[i[ok]]
    return np.where(ok, i, -1)
def in_bouts(t, mt):
    return bout_of(np.asarray(t), mt) >= 0
def within_trial_freq(times, bout=None):
    """Inverse intervals between consecutive onsets in the same trial; with bout, both onsets in the
    same lick bout (round 1 corrected this: pairs in two adjacent bouts were counted)."""
    t = np.asarray(times)
    d = np.diff(t)
    same = trial_of(t[:-1]) == trial_of(t[1:])
    if bout is not None:
        same &= (bout[:-1] >= 0) & (bout[:-1] == bout[1:])
    return 1 / d[same & (d > 0)]
lf_a, bf_a, lf_b, bf_b, lf_x = [], [], [], [], []
n_cross = 0
for r in res:
    mt = r["mt"]
    ton, ins = mt["tongue_onset_all"], mt["inspir_onset"]
    bt, bi = bout_of(ton, mt), bout_of(ins, mt)
    lf_a.append(within_trial_freq(ton)); bf_a.append(within_trial_freq(ins))
    lf_b.append(within_trial_freq(ton, bt)); bf_b.append(within_trial_freq(ins, bi))
    # the round-1 version: both onsets in some bout, not necessarily the same one
    inb = bt >= 0
    same = (trial_of(ton[:-1]) == trial_of(ton[1:])) & inb[:-1] & inb[1:]
    lf_x.append(1 / np.diff(ton)[same & (np.diff(ton) > 0)])
    n_cross += int((same & (bt[:-1] != bt[1:])).sum())
rows = []
for name, lf, bf in (("(a) within trials", lf_a, bf_a), ("(b) within lick bouts", lf_b, bf_b)):
    L, B = np.hstack(lf), np.hstack(bf)
    rows.append({"reading": name, "lick mean": L.mean(), "lick sd": L.std(), "lick 2.5%": np.percentile(L, 2.5), "lick 97.5%": np.percentile(L, 97.5),
                 "breath mean": B.mean(), "breath sd": B.std(), "breath 2.5%": np.percentile(B, 2.5), "breath 97.5%": np.percentile(B, 97.5),
                 "n licks": len(L), "n breaths": len(B)})
fq = pd.DataFrame(rows)
print(fq.round(2).to_string(index=False))
exp = {"lick mean": (6.23, .62), "breath mean": (4.02, .4), "lick 2.5%": (3.59, .36), "lick 97.5%": (8.91, .89), "breath 2.5%": (2.12, .21), "breath 97.5%": (7.17, .72)}
for _, r in fq.iterrows():
    print(r.reading, {k: bool(abs(r[k] - v) <= t) for k, (v, t) in exp.items()})
print("\nPer session median frequencies, reading (b): licking", np.round(np.percentile([np.median(x) for x in lf_b if len(x)], [5, 50, 95]), 2),
      "breathing", np.round(np.percentile([np.median(x) for x in bf_b if len(x)], [5, 50, 95]), 2), "(5th, 50th, 95th percentiles over sessions)")
# Diagnostics asked for by the verifier (round 1); they do not decide the outcome.
LX, LA = np.hstack(lf_x), np.hstack(lf_a)
two_frames = 1 / (2 * kaku.FRAME_S)
print(f"\nRound-1 version of (b) (onset pairs in two adjacent bouts counted): {len(LX)} intervals, {n_cross} of them between two bouts; "
      f"lick mean {LX.mean():.2f} Hz, 97.5% {np.percentile(LX, 97.5):.2f} Hz")
print(f"Lick intervals shorter than 2 frames (6.8 ms, > {two_frames:.0f} Hz): {(LA > two_frames).sum()} of {len(LA)} under (a), "
      f"{(np.hstack(lf_b) > two_frames).sum()} of {len(np.hstack(lf_b))} under (b)")
LA2 = LA[LA <= two_frames]
print(f"(a) without intervals under 2 frames: lick mean {LA2.mean():.2f} Hz, 2.5% {np.percentile(LA2, 2.5):.2f}, 97.5% {np.percentile(LA2, 97.5):.2f} Hz (n = {len(LA2)})")

**Result: partially reproduced.** Under reading (b), within lick bouts, licking is 6.07 Hz (95% range
3.77 to 8.40 Hz) against 6.23 (3.59 to 8.91), and breathing 4.19 Hz against 4.02: both means and both
licking bounds are within tolerance, but the breathing bounds (2.41 and 6.39 Hz against 2.12 and 7.17)
are not. Under reading (a), within trials, breathing matches on all three values (3.89 Hz, 2.10 to 6.68)
but licking does not (26.96 Hz). Removing the 39,027 intervals under 2 frames leaves a mean of 21.36 Hz
(97.5th percentile 98 Hz), so those intervals do not explain the high frequency under (a). Round 1 counted, under (b), onset pairs that lie in two
adjacent bouts (13,498 such pairs), which gave 8.06 Hz; reading (b) now takes only pairs inside one bout. The code builds bouts from runs of lick intervals between 3
and 9 Hz (`kaku.movement_timing`), so the licking mean and range under (b) are largely fixed by that
definition; the breathing values are not.

In [ ]:
rows = []
lat_by_n = []
for r, (_, s) in zip(res, sess.iterrows()):
    mt = r["mt"]
    ton = mt["tongue_onset_all"]
    # protrusion intervals: each tongue onset to the first offset after it (code onsets and offsets)
    off = np.sort(mt["tongue_offset"])
    j = np.searchsorted(off, ton, side="right")
    ok = j < len(off)
    on_i, off_i = ton[ok], off[j[ok]]
    ins = mt["inspir_onset"]
    ins_b = ins[in_bouts(ins, mt)]
    k = np.searchsorted(on_i, ins_b, side="right") - 1
    inside = (k >= 0) & (ins_b <= off_i[np.clip(k, 0, None)]) & (ins_b >= on_i[np.clip(k, 0, None)])
    rows.append({"subject": s.subject, "session": s.session, "inspirations in bouts": len(ins_b), "within protrusion": int(inside.sum())})
pr = pd.DataFrame(rows)
pr["fraction"] = pr["within protrusion"] / pr["inspirations in bouts"]
print("Per session fraction of in-bout inspiration onsets that fall inside a tongue protrusion:")
print(pr.fraction.describe().round(4).to_string())
pooled = pr["within protrusion"].sum() / pr["inspirations in bouts"].sum()
print(f"\nPooled: {pr['within protrusion'].sum()} of {pr['inspirations in bouts'].sum()} = {pooled:.4f}; reading (b) holds (< 5%): {pooled < 0.05}")
print("Reading (a), no such inspiration in any session:", bool((pr['within protrusion'] == 0).all()), "; sessions with none:", int((pr['within protrusion'] == 0).sum()), "of", len(pr))
print("By subject (pooled fraction):")
print((pr.groupby("subject")["within protrusion"].sum() / pr.groupby("subject")["inspirations in bouts"].sum()).round(4).to_string())
# Per subject, DL029 to DL034 apart (verifier, round 2), beside the airflow-tongue lag of the data check. Variant:
# each subject's inspiration onsets moved earlier by its lag minus the median lag of the other subjects
# that peak within the first cycle, as if the extra lag were a clock offset (subjects peaking at the next
# cycle are not moved). This is a check of the alignment question, not a reading of the claim.
lag_t = XCS[XCS.signal == "tongue"].set_index("subject")["lag of extreme (ms)"]
ref = lag_t[lag_t.index.isin(first) & ~lag_t.index.isin(LOW)].median()
shift = {sub: (l - ref) / 1000 if sub in set(first) else 0.0 for sub, l in lag_t.items()}
def inside_fraction(r, dt):
    mt = r["mt"]
    ton = mt["tongue_onset_all"]
    off = np.sort(mt["tongue_offset"])
    j = np.searchsorted(off, ton, side="right")
    ok = j < len(off)
    on_i, off_i = ton[ok], off[j[ok]]
    ins = mt["inspir_onset"] - dt
    ins_b = ins[in_bouts(ins, mt)]
    k = np.searchsorted(on_i, ins_b, side="right") - 1
    inside = (k >= 0) & (ins_b <= off_i[np.clip(k, 0, None)]) & (ins_b >= on_i[np.clip(k, 0, None)])
    return int(inside.sum()), len(ins_b)
sh = pd.DataFrame([(s.subject, *inside_fraction(r, shift.get(s.subject, 0.0))) for r, (_, s) in zip(res, sess.iterrows())],
                  columns=["subject", "inside", "n"]).groupby("subject").sum()
PS = pd.DataFrame({"group": ["DL029-DL034" if s in LOW else "other" for s in sorted(set(pr.subject))],
                   "fraction": pr.groupby("subject")["within protrusion"].sum() / pr.groupby("subject")["inspirations in bouts"].sum(),
                   "tongue lag (ms)": lag_t, "shift (ms)": pd.Series(shift) * 1000, "fraction after shift": sh.inside / sh.n})
print(f"\nPer subject (shift: subject lag minus {ref:.1f} ms, the median of the other first-cycle subjects):")
print(PS.round(4).to_string())
print(PS.groupby("group")[["fraction", "fraction after shift"]].agg(["min", "median", "max"]).round(4).to_string())
fsub = PS[PS.index.isin(first)]
rho = stats.spearmanr(fsub["tongue lag (ms)"], fsub.fraction)
print(f"Spearman, fraction against tongue lag over the {len(fsub)} first-cycle subjects: rho = {rho.correlation:.2f}, p = {rho.pvalue:.2g}")
print(f"Pooled after the shift: {sh.inside.sum()} of {sh.n.sum()} = {sh.inside.sum() / sh.n.sum():.4f}")

# latency from the last lick to the next inspiration, per number of licks in the cycle (code cycles)
lat = []
for r in res:
    p = r["pd_lick"]
    if p is None:
        continue
    lat.append(pd.DataFrame({"n_licks": p["n_licks"], "lat": p.get("last_lick_to_next_insp", np.full(len(p["n_licks"]), np.nan)), "ibi": p["ibi"]}))
lat = pd.concat(lat)
print("\nLatency from the last lick in a breath cycle to the next inspiration (s), pooled:")
print(lat.groupby("n_licks").lat.describe()[["count", "25%", "50%", "75%"]].round(3).head(6).to_string())

**Result: not reproduced.** 30.4% of in-bout inspiration onsets (70,298 of 231,563) fall between a
tongue onset and the following tongue offset, above the 5% of reading (b); every session has some
(3.3% to 46.1% per session; by subject 8.5% to 42.5%). Both readings fail. The latency from the last
lick of a cycle to the next inspiration has a median of 133 ms after one lick and 92 ms after two
(pooled). Per subject, the fraction is lower in DL029 to DL034 (8.5% to 14.6%) than in the others (22% to
42%), and over the 28 subjects whose airflow-tongue correlation peaks within the first cycle it falls
as that lag rises (Spearman rho = -0.67, p = 0.00011). The data check shows a later airflow lag in exactly
these subjects. Moving each subject's inspirations earlier by its lag minus the 78.2 ms median of the
others raises their fractions to 10.3% to 21.0% and leaves the pooled fraction at 30.7%, so the
difference between subjects is consistent with a clock offset of 20 to 30 ms in DL029 to DL034, and no
correction for it brings the fraction near 0 or 5%.

In [ ]:
pdl = pd.DataFrame([{"subject": s.subject, "session": s.session, "diff": (r["pd_lick"] or {}).get("diff", np.nan),
                     "n_fast": len((r["pd_lick"] or {}).get("i_fast", [])), "n_slow": len((r["pd_lick"] or {}).get("i_slow", []))}
                    for r, (_, s) in zip(res, sess.iterrows())])
v = pdl.dropna(subset=["diff"])
print(f"Sessions with at least 40 two-lick cycles in each half and a peak in each window: {len(v)} of {len(pdl)} "
      f"({((pdl.n_fast >= 40) & (pdl.n_slow >= 40)).sum()} with enough cycles)")
print("Session peak differences, slow minus fast (s):", v["diff"].describe().round(4).to_dict())
t_a = stats.ttest_1samp(v["diff"], 0)
m = v.groupby("subject")["diff"].mean()
t_b = stats.ttest_1samp(m, 0)
print(f"(a) sessions: mean {v['diff'].mean() * 1000:.1f} ms, t = {t_a.statistic:.2f}, p = {t_a.pvalue:.2e}, n = {len(v)}")
print(f"(b) mice: mean {m.mean() * 1000:.1f} ms, t = {t_b.statistic:.2f}, p = {t_b.pvalue:.2e}, n = {len(m)}")
print("Mice with a positive mean:", int((m > 0).sum()), "of", len(m), "; zero:", int((m == 0).sum()), "; negative:", int((m < 0).sum()))
print("Per mouse mean (ms) and sessions:")
print(pd.DataFrame({"mean_ms": (m * 1000).round(1), "sessions": v.groupby("subject").size()}).T.to_string())
print("Sessions with difference > 0:", int((v['diff'] > 0).sum()), "; = 0:", int((v['diff'] == 0).sum()), "; < 0:", int((v['diff'] < 0).sum()))

def null_report(key, data):
    """Random-split null (verifier, round 1). The fast and slow windows differ, so a peak between their
    starts gives a positive difference with no effect. Per session, the difference for 20 random splits of
    the same cycles into groups of the halves' sizes (kaku.random_split); the session's null value is
    their mean. Data against null, paired over the sessions that have both."""
    rows = []
    for r, (_, s) in zip(res, sess.iterrows()):
        p = r[key] or {}
        nul = np.asarray(p.get("diff_null", []), float)
        rows.append({"subject": s.subject, "session": s.session, "diff": p.get("diff", np.nan),
                     "null": np.nanmean(nul) if np.isfinite(nul).any() else np.nan,
                     "null_first": nul[np.isfinite(nul)][0] if np.isfinite(nul).any() else np.nan,
                     "null_pos": np.mean(nul[np.isfinite(nul)] > 0) if np.isfinite(nul).any() else np.nan})
    N = pd.DataFrame(rows)
    both = N.dropna(subset=["diff", "null"])
    t0 = stats.ttest_1samp(N.null_first.dropna(), 0)
    tp = stats.ttest_rel(both["diff"], both["null"])
    print(f"Null, one random split per session: {N.null_first.notna().sum()} sessions, mean {N.null_first.mean() * 1000:.1f} ms, p = {t0.pvalue:.2e} (t-test against 0)")
    print(f"Null, mean of 20 splits per session: mean {N.null.mean() * 1000:.1f} ms; share of splits above 0, median over sessions {N.null_pos.median():.2f}")
    print(f"Data minus null over {len(both)} sessions: mean {(both['diff'] - both['null']).mean() * 1000:.1f} ms, paired t-test p = {tp.pvalue:.2e}; "
          f"data above null in {int((both['diff'] > both['null']).sum())}")
    mm = both.groupby("subject")[["diff", "null"]].mean()
    tm = stats.ttest_rel(mm["diff"], mm["null"]) if len(mm) > 2 else None
    print(f"Mice: data above null in {int((mm['diff'] > mm['null']).sum())} of {len(mm)}; paired t-test p = {tm.pvalue if tm else np.nan:.2e}")
    return N
NL1g = null_report("pd_lick", v)
# DL029 to DL034 apart (verifier, round 2)
grp = np.where(NL1g.subject.isin(LOW), "DL029-DL034", "other")
G = NL1g.assign(group=grp).dropna(subset=["diff"]).groupby("group").agg(sessions=("diff", "size"), mice=("subject", "nunique"),
                                                                          mean_ms=("diff", "mean"), null_ms=("null", "mean"))
G[["mean_ms", "null_ms"]] *= 1000
print(G.round(1).to_string())

**Result: partially reproduced.** Over the 71 sessions with enough two-lick cycles and a peak in
each window, the second lick is later before a slow breath by 22.0 ms on average (t = 3.68, p =
4.5e-4, reading (a)); over the 20 mice the mean is 13.6 ms (p = 0.18, reading (b)). 16 of the 20 mice
have a positive mean, one zero and three negative, so not every mouse is positive. 102 of 173 sessions
give no value (fewer than 40 cycles in a half, or no peak). The random-split null, run in round 1 of
verification, gives a positive difference too: 15.7 ms over 83 sessions with one random split each
(p = 0.0072), and the data exceed the mean of 20 random splits by 13.3 ms (paired p = 0.037 over 71
sessions, 54 of them above the null; over mice 14 of 20 above it, paired p = 0.98). The slow window of
the header starts 40 ms after the fast one, so a positive difference arises without an effect, and on
these data the measured difference is separated from it over sessions but not over mice. Of DL029 to
DL034 only DL031 has a value (one session, 0 ms); the other 70 sessions in 19 mice give 22.3 ms. The outcome follows the claims file's criterion, which compares with 0.

In [ ]:
rows = []
for r, (_, s) in zip(res, sess.iterrows()):
    p = r["pd_lick"]
    if p is None:
        continue
    d = pd.DataFrame({"n": p["n_licks"], "ibi": p["ibi"], "lat": p.get("last_lick_to_next_insp", np.full(len(p["ibi"]), np.nan))})
    g = d.groupby("n").agg(med_ibi=("ibi", "median"), med_lat=("lat", "median"), count=("ibi", "size"))
    row = {"subject": s.subject, "session": s.session}
    for n in (1, 2, 3, 4):
        row[f"ibi{n}"] = g.med_ibi.get(n, np.nan); row[f"lat{n}"] = g.med_lat.get(n, np.nan); row[f"n{n}"] = int(g["count"].get(n, 0))
    rows.append(row)
ms = pd.DataFrame(rows)
ok3 = (ms.n1 > 0) & (ms.n2 > 0) & (ms.n3 > 0)
rise = (ms.ibi1 < ms.ibi2) & (ms.ibi2 < ms.ibi3)
rise4 = rise & ((ms.n4 < 20) | (ms.ibi3 < ms.ibi4))
print(f"Sessions with cycles of 1, 2 and 3 licks: {ok3.sum()} of {len(ms)}")
print(f"Median inter-breath interval rises from 1 to 2 to 3 licks (and to 4 where >= 20 cycles): {int((rise4 & ok3).sum())} of {int(ok3.sum())} sessions; "
      f"without the 4-lick condition {int((rise & ok3).sum())}")
print("Median over sessions of the session medians (s):", ms.loc[ok3, ["ibi1", "ibi2", "ibi3", "ibi4"]].median().round(3).to_dict())
lat_med = lat.groupby("n_licks").lat.median()
print("Pooled median latency last lick -> next inspiration (s) by licks in cycle:", lat_med.head(4).round(4).to_dict())
l4 = lat_med.loc[[1, 2, 3]]
print(f"Largest relative difference between groups 1 to 3: {(l4.max() - l4.min()) / l4.min():.1%}; with group 4: {(lat_med.loc[[1, 2, 3, 4]].max() - lat_med.loc[[1, 2, 3, 4]].min()) / lat_med.loc[[1, 2, 3, 4]].min():.1%}")
sl = ms.loc[ok3, ["lat1", "lat2", "lat3"]]
rel = (sl.max(1) - sl.min(1)) / sl.min(1)
print(f"Per session, groups 1 to 3 within 20% of each other: {int((rel < 0.2).sum())} of {len(sl)}")
print("Cycles per group, pooled:", lat.groupby('n_licks').size().head(6).to_dict())

**Result: not reproduced.** The median inter-breath interval rises from 1 to 2 to 3 licks per cycle
in 116 of 117 sessions (medians over sessions 0.228, 0.296 and 0.418 s), so licks are added as
breathing slows. The latency from the last lick to the next inspiration is not fixed by the criterion's
20% rule: pooled medians are 133, 92 and 82 ms for 1, 2 and 3 licks (62.5% apart), and only 1 of 117
sessions keeps them within 20%. The criterion asks for both parts and defines no partial outcome.

In [ ]:
pdb = pd.DataFrame([{"subject": s.subject, "session": s.session, "diff": (r["pd_breath"] or {}).get("diff", np.nan),
                     "n_fast": len((r["pd_breath"] or {}).get("i_fast", []))} for r, (_, s) in zip(res, sess.iterrows())])
v2 = pdb.dropna(subset=["diff"])
print(f"Sessions with at least 100 cycles per half and a peak in each window: {len(v2)} of {len(pdb)}")
print("Session peak differences (s):", v2["diff"].describe().round(4).to_dict())
t_a2 = stats.ttest_1samp(v2["diff"], 0)
m2 = v2.groupby("subject")["diff"].mean()
t_b2 = stats.ttest_1samp(m2, 0)
print(f"(a) sessions: mean {v2['diff'].mean() * 1000:.1f} ms, t = {t_a2.statistic:.2f}, p = {t_a2.pvalue:.2e}, n = {len(v2)}")
print(f"(b) mice: mean {m2.mean() * 1000:.1f} ms, t = {t_b2.statistic:.2f}, p = {t_b2.pvalue:.2e}, n = {len(m2)}")
print("Mice with a positive mean:", int((m2 > 0).sum()), "of", len(m2))
print("Sessions > 0:", int((v2['diff'] > 0).sum()), "; = 0:", int((v2['diff'] == 0).sum()), "; < 0:", int((v2['diff'] < 0).sum()))
NL1i = null_report("pd_breath", None)

**Result: reproduced.** Over 86 sessions the inspiration after a lick is later before a slow lick by
10.5 ms on average (t = 6.79, p = 1.5e-9, reading (a)); over 23 mice 9.6 ms (p = 0.0012, reading (b)).
44 sessions are positive, 37 zero and 5 negative. The random-split null gives 2.7 ms (one split per
session, p = 0.12), and the data exceed the mean of 20 random splits by 10.6 ms (paired p = 2.1e-10;
19 of 23 mice above the null), so the effect is not an artefact of the two windows.

In [ ]:
# Extended Data Fig 1d-i: unidirectional models, simulated as the Methods describe.
# Inter-breath intervals come from fig1d reading (a) and inter-lick intervals from reading (b): for each
# rhythm the reading whose values are closest to the paper's Fig 1d (fig1d above).
rng = np.random.default_rng(11)
ibi_emp = 1 / np.hstack(bf_a)          # inter-breath intervals within trials (fig1d reading (a))
ili_emp = 1 / np.hstack(lf_b)          # inter-lick intervals within one lick bout (fig1d reading (b))
ili_range = np.percentile(ili_emp, [2.5, 97.5])
ibi_range = np.percentile(ibi_emp, [2.5, 97.5])
lat_ll = lat.lat.median() if lat.lat.notna().any() else np.nan       # last lick -> next inspiration
# last inspiration -> next lick, from the breath-to-lick cycles of the data
lat_il = []
for r in res:
    mt = r["mt"]
    ton, ins = mt["tongue_onset"], mt["inspir_onset"]
    tb = ton[in_bouts(ton, mt)]
    k = np.searchsorted(ins, tb) - 1
    ok = k >= 0
    lat_il.append(tb[ok] - ins[k[ok]])
lat_il = np.median(np.hstack(lat_il))
print(f"Empirical ranges: inter-lick {ili_range.round(3)} s, inter-breath {ibi_range.round(3)} s; latencies: last lick -> inspiration {lat_ll:.3f} s, inspiration -> next lick {lat_il:.3f} s")

def sim_breath_driven(n_breaths):
    ins = np.cumsum(rng.choice(ibi_emp, n_breaths))
    licks = np.sort(ins - lat_ll)
    out = [licks[0]]
    for a, b in zip(licks[:-1], licks[1:]):
        gap = b - a
        k = int(np.ceil(gap / ili_range[1])) - 1 if gap > ili_range[1] else 0
        out += list(a + gap * np.arange(1, k + 1) / (k + 1)) + [b]
    licks = np.sort(np.array(out) + rng.normal(0, 0.01, len(out)))
    return licks, ins

def sim_lick_driven(n_licks):
    licks = np.cumsum(rng.choice(ili_emp, n_licks))
    cand = licks - lat_il
    ins = [cand[0]]
    for c in cand[1:]:
        if ibi_range[0] <= c - ins[-1]:
            ins.append(c)
    ins = np.array(ins) + rng.normal(0, 0.01, len(ins))
    return licks, np.sort(ins)

def measures(licks, ins):
    mt = {"tongue_onset": licks, "inspir_onset": ins, "bout_on": np.array([licks[0]]), "bout_off": np.array([licks[-1]])}
    a = kaku.peak_diff_lick_behv(mt) or {}
    b = kaku.peak_diff_breath_behv(mt) or {}
    return {"fig1g": a.get("diff", np.nan), "fig1i": b.get("diff", np.nan),
            "two-lick fast": len(a.get("i_fast", [])), "two-lick slow": len(a.get("i_slow", [])),
            "fast peak": len(a.get("peaks_fast", [])) > 0, "slow peak": len(a.get("peaks_slow", [])) > 0}

sim, diag = [], []
for r in res:
    mt = r["mt"]
    nb = int(in_bouts(mt["inspir_onset"], mt).sum()); nl = int(in_bouts(mt["tongue_onset"], mt).sum())
    if nb < 50:
        continue
    mb = measures(*sim_breath_driven(nb))
    ml = measures(*sim_lick_driven(nl))
    sim.append({"breath_model_fig1g": mb["fig1g"], "breath_model_fig1i": mb["fig1i"], "lick_model_fig1g": ml["fig1g"], "lick_model_fig1i": ml["fig1i"]})
    diag.append({**{f"breath model {k}": v for k, v in mb.items() if k not in ("fig1g", "fig1i")},
                 **{f"lick model {k}": v for k, v in ml.items() if k not in ("fig1g", "fig1i")}})
sim = pd.DataFrame(sim); diag = pd.DataFrame(diag)
for col in sim.columns:
    v_ = sim[col].dropna()
    tt = stats.ttest_1samp(v_, 0) if len(v_) > 2 and v_.std() > 0 else None
    print(f"{col}: sessions with a value {len(v_)} of {len(sim)}, mean {v_.mean() * 1000 if len(v_) else np.nan:.1f} ms, "
          f"SD {v_.std() * 1000 if len(v_) > 1 else np.nan:.1f} ms, p = {tt.pvalue if tt else np.nan:.3g}")
print(f"Data: Fig 1g mean {v['diff'].mean() * 1000:.1f} ms (p = {t_a.pvalue:.2e}); Fig 1i mean {v2['diff'].mean() * 1000:.1f} ms (p = {t_a2.pvalue:.2e})")
# Why the Fig 1g measure is mostly missing in the licking-driven model (verifier, round 1)
for mdl in ("lick model", "breath model"):
    nf, ns = diag[f"{mdl} two-lick fast"], diag[f"{mdl} two-lick slow"]
    enough = (nf >= 40) & (ns >= 40)
    print(f"{mdl}: two-lick cycles per half, median {nf.median():.0f} (fast) and {ns.median():.0f} (slow); sessions with >= 40 in both "
          f"{int(enough.sum())} of {len(diag)}; of those, a peak in the fast window {int(diag.loc[enough, f'{mdl} fast peak'].sum())}, "
          f"in the slow window {int(diag.loc[enough, f'{mdl} slow peak'].sum())}")

**Result: reproduced.** With the data's interval distributions (inter-breath intervals from fig1d
reading (a), inter-lick intervals from reading (b)) and latencies (112 ms last lick to inspiration,
119 ms inspiration to next lick), the breathing-driven model gives a Fig 1i difference of 0.0 ms in all
162 simulated sessions with a value (no effect). The licking-driven model gives a Fig 1g value in only
1 of 172 sessions: 145 sessions have at least 40 two-lick cycles per half, but the lick PSTH has a peak
above threshold in the fast window in 18 of them and in the slow window in 12, since simulated licks do
not lock to inspiration. So neither model produces the relation it lacks, while the data give both
differences positive (22.0 and 10.5 ms, p < 0.001). The licking-driven result holds because the measure
is missing, not because a test found no effect. Each model does produce the other relation
(breathing-driven Fig 1g 59.3 ms; licking-driven Fig 1i 35.9 ms).

In [ ]:
# Extended Data Fig 1b-c: lick probability around swallows and around expirations inside bouts
bins = np.arange(-0.3, 0.3 + 1e-9, kaku.FRAME_S)
rows, tr_sw, tr_ex = [], [], []
for r, (_, s) in zip(res, sess.iterrows()):
    sw = r["swallow_times"]
    if len(sw) < 20:
        continue
    mt = r["mt"]
    ton = mt["tongue_onset_all"]
    ex = r["expir_onset"]
    ex = ex[in_bouts(ex, mt)]
    def prob(ev):
        h = np.zeros(len(bins) - 1)
        for e in ev:
            h += np.histogram(ton - e, bins)[0]
        return h / max(len(ev), 1)
    ps, pe = prob(sw), prob(ex)
    tr_sw.append(ps); tr_ex.append(pe)
    c = bins[:-1]
    w = (c >= 0) & (c < 0.1)
    rows.append({"subject": s.subject, "session": s.session, "swallows": len(sw), "expirations": len(ex),
                 "p_swallow": ps[w].mean(), "p_expiration": pe[w].mean()})
lp = pd.DataFrame(rows)
wt = stats.wilcoxon(lp.p_swallow, lp.p_expiration)
print(f"Sessions with >= 20 swallows: {len(lp)}; lick probability per 3.4 ms bin, 0 to 100 ms after onset:")
print(lp[["p_swallow", "p_expiration"]].describe().round(4).to_string())
print(f"Lower around swallows in {int((lp.p_swallow < lp.p_expiration).sum())} of {len(lp)} sessions; Wilcoxon signed-rank p = {wt.pvalue:.2e}")
print("Median ratio swallow / expiration:", round(float((lp.p_swallow / lp.p_expiration).median()), 3))
# The whole trace from -300 to 300 ms (verifier, round 1): mean over sessions, in 30 ms steps (each the
# mean of the 3.4 ms bins whose start falls in it)
c = bins[:-1]
steps = np.arange(-0.3, 0.3, 0.03)
TS, TE = np.mean(tr_sw, 0), np.mean(tr_ex, 0)
trace = pd.DataFrame({"from (ms)": (steps * 1000).round(0).astype(int),
                      "swallow": [TS[(c >= a) & (c < a + 0.03)].mean() for a in steps],
                      "expiration": [TE[(c >= a) & (c < a + 0.03)].mean() for a in steps]})
print("\nLick probability per 3.4 ms bin, mean over sessions:")
print(trace.round(4).T.to_string(header=False))
print(f"Minimum of the swallow trace: {TS.min():.4f} at {c[np.argmin(TS)] * 1000:.0f} ms; of the expiration trace: {TE.min():.4f} at {c[np.argmin(TE)] * 1000:.0f} ms")

**Result: not reproduced.** In 90 sessions with at least 20 swallows, the probability of a tongue
onset per 3.4 ms bin in the 100 ms after swallow onset is higher than after in-bout expiration onsets
(mean 0.0308 against 0.0211; lower around swallows in 4 of 90 sessions; Wilcoxon p = 4.0e-16). The full
trace shows a pause before the onset: from -90 to 0 ms the swallow trace is 0.0036 to 0.0068 per bin
(minimum 0.0026 at -48 ms) against about 0.021 around expirations, and licking resumes from 0 to 60 ms
(0.030 and 0.050). The code's detector requires a lick within 0.25 s after the detected peak and takes
the onset at the start of the airflow pause, so the criterion's window after onset falls where licking
resumes. This is put to a reader as a question on the criterion's window.

In [ ]:
# Figure 1k-n: swallow-aligned licks and breaths (code get_licks_around_event, plot_*_swallow_summary)
al_l, al_b, owner = [], [], []
for r, (_, s) in zip(res, sess.iterrows()):
    mt = r["mt"]
    a = kaku.events_around_swallows(r["swallows"], mt["tongue_onset"], r["trial_len"])
    b = kaku.events_around_swallows(r["swallows"], mt["inspir_onset"], r["trial_len"])
    al_l += a; al_b += b; owner += [s.session] * len(a)
owner = np.array(owner)
SWL = kaku.swallow_intervals(al_l, max_ili=1 / 3)
SWB = kaku.swallow_intervals(al_b, max_ili=1 / 2)
SWL["session"] = owner[SWL.event]; SWB["session"] = owner[SWB.event]
print("Swallow events pooled:", len(al_l), "; kept for licks:", len(SWL), "; kept for breaths:", len(SWB))

def readings(df):
    n3 = len(df) // 3
    out = {"(a) code: longest third of cycle 0": df.iloc[:n3],
           "(b) longest third of cycle -1": df.sort_values("cm1", ascending=False).iloc[:n3],
           "(c) all events": df}
    return out

def k_test(df, name):
    n3 = len(df) // 3
    for rk, col in (("(a) by cycle 0", "c0"), ("(b) by cycle -1", "cm1")):
        o = df.sort_values(col, ascending=False)
        top, bot = o.iloc[:n3], o.iloc[-n3:]
        dl = abs(top.latency.median() - bot.latency.median()); dc = abs(top.cm1.median() - bot.cm1.median())
        print(f"{name} {rk}: median latency to first {name} after swallow, top third {top.latency.median() * 1000:.1f} ms, bottom third {bot.latency.median() * 1000:.1f} ms "
              f"(difference {dl * 1000:.1f}); median cycle -1 {top.cm1.median() * 1000:.1f} and {bot.cm1.median() * 1000:.1f} ms (difference {dc * 1000:.1f}); smaller: {dl < dc}")
k_test(SWL, "lick")
k_test(SWB, "breath")

**Result: reproduced.** For licks and for breaths, under both rankings, the median latency from
swallow onset to the next event differs less between the top and bottom thirds (licks 17.0 ms;
breaths 37.4 and 23.8 ms) than their median cycle -1 does (licks 27.2 and 40.8 ms; breaths 95.2 and
122.4 ms).

In [ ]:
def delta_tests(df, label):
    rows = []
    for name, d in readings(df).items():
        for q, col in (("Delta(1,-1)", "d1"), ("Delta(0,-1)", "d0")):
            x = (d.c1 - d.cm1) if col == "d1" else (d.c0 - d.cm1)
            t = stats.ttest_1samp(x, 0)
            rows.append({"events": label, "reading": name, "quantity": q, "n": len(x), "mean_ms": x.mean() * 1000, "p": t.pvalue})
    d = readings(df)["(a) code: longest third of cycle 0"]
    for q in ("Delta(1,-1)", "Delta(0,-1)"):
        x = (d.c1 - d.cm1) if q == "Delta(1,-1)" else (d.c0 - d.cm1)
        sm = x.groupby(d.session.to_numpy()).mean()
        t = stats.ttest_1samp(sm, 0)
        rows.append({"events": label, "reading": "(d) (a) with session means", "quantity": q, "n": len(sm), "mean_ms": sm.mean() * 1000, "p": t.pvalue})
    return pd.DataFrame(rows)
DL = delta_tests(SWL, "licks")
print(DL.round(4).to_string(index=False))

def cycle_report(df, max_c0, label):
    """Distributions of cycles -1, 0 and 1 under O5 (a) (verifier, round 1). The code limits cycle 0 to
    max_c0 and sets no limit on cycle 1, so a pause after the swallow enters Delta(1,-1)."""
    d = readings(df)["(a) code: longest third of cycle 0"]
    q = pd.DataFrame({k: (d[c] * 1000).quantile([0.05, 0.25, 0.5, 0.75, 0.95]) for k, c in (("cycle -1", "cm1"), ("cycle 0", "c0"), ("cycle 1", "c1"))})
    print(f"\n{label}, O5 (a), cycle quantiles (ms):")
    print(q.round(1).T.to_string())
    long1 = d.c1 > max_c0
    print(f"Cycle 1 longer than {max_c0 * 1000:.0f} ms (the cycle-0 limit): {int(long1.sum())} of {len(d)} ({long1.mean():.1%}); "
          f"cycle -1 longer: {int((d.cm1 > max_c0).sum())}")
    x = (d.c1 - d.cm1)[~long1]
    t = stats.ttest_1samp(x, 0)
    print(f"Diagnostic, not a reading: Delta(1,-1) without those events {x.mean() * 1000:.1f} ms (n = {len(x)}, p = {t.pvalue:.2e}); "
          f"median Delta(1,-1) over all events {(d.c1 - d.cm1).median() * 1000:.1f} ms")
cycle_report(SWL, 1 / 3, "Licks")

**Result: fig1l-lick-cycle-prolonged reproduced; fig1l-lick-interval-preserved not reproduced.** On
the code's selection (reading (a), 1,932 events), the lick cycle spanning the swallow is longer than
the cycle before by 33.31 ms (p < 0.0001), against 33.3 ms. The cycle after the swallow is also longer
than the one before, by 119.5 ms (p < 0.0001), so the paper's test finds the effect the claim says is
absent; readings (b) to (d) give 86.2, 50.3 and 87.3 ms, all p < 0.001. The mean is driven by long
pauses after the swallow: cycle 1 exceeds 333 ms in 270 of 1,932 events (95th percentile 1,256 ms) and
the median Delta(1,-1) is 0.0 ms. Without those events (a diagnostic, not a reading) the mean is
-16.3 ms, p = 3e-24, which still differs from 0. The port computes cycle 1 as the paper's code does
(generate_figures.plot_licking_swallow_summary, after_2 - after_1, with no limit on cycle 1).

In [ ]:
DB = delta_tests(SWB, "breaths")
print(DB.round(4).to_string(index=False))
cycle_report(SWB, 1 / 2, "Breaths")

**Result: both reproduced.** On the code's selection (2,100 events), the breath cycle spanning the
swallow is longer than the cycle before by 48.02 ms (p < 0.0001), against 48.0 ms, and the cycle after
differs from the one before by -2.1 ms (p = 0.20), so the test does not find a change. Reading (c)
agrees (-0.9 ms, p = 0.26); readings (b) and (d) find a difference (-25.3 ms, p < 0.001; -6.6 ms, p =
0.0099).

## Phase tuning (Figure 2, Extended Data Figures 4, 5 and 8)

The measured MI is the code's curve_fit; significance uses the grid fit (helper docstring). The
check below compares the two fits, and the code's own 100-resample scheme with curve_fit against
the grid version on a random sample of units.

In [ ]:
rng_chk = np.random.default_rng(5)
chk = []
pool = [(r, p, u) for r in res for p, us in r["units"].items() for u in us]
idx = rng_chk.choice(len(pool), 300, replace=False)
for i in idx:
    r, p, u = pool[i]
    for mod, fs in (("jaw", r["video_fs"]), ("br", 250.0)):
        cnt, occ, cen = u[f"{mod}_cnt_5"], u[f"{mod}_occ_5"], u[f"{mod}_cen_5"]
        if cnt is None or cnt.sum() == 0 or (occ == 0).any():
            continue
        chk.append({"mod": mod, "code": u[f"{mod}_mi_5"], "grid": u[f"{mod}_mi_grid_5"]})
chk = pd.DataFrame(chk)
for mod, g in chk.groupby("mod"):
    d = (g.grid - g.code).abs()
    print(f"{mod}: {len(g)} random units, |grid MI - code MI| median {d.median():.4f}, 90th pct {d.quantile(.9):.3f}; "
          f"same side of 0.6 in {np.mean((g.grid > 0.6) == (g.code > 0.6)):.3f}")
# the code's significance scheme with curve_fit (100 + 100 resamples) on units with code MI > 0.6
cand = [(r, p, u, mod) for (r, p, u) in pool for mod in ("jaw", "br") if (u.get(f"{mod}_mi_5") or 0) > 0.6]
sel = rng_chk.choice(len(cand), 24, replace=False)
rows = []
t0 = time.time()
for i in sel:
    r, p, u, mod = cand[i]
    fs = r["video_fs"] if mod == "jaw" else 250.0
    cnt, occ, cen = u[f"{mod}_cnt_5"], u[f"{mod}_occ_5"], u[f"{mod}_cen_5"]
    n = int(cnt.sum())
    boot = rng_chk.multinomial(n, cnt / n, size=100) / occ * fs
    null = rng_chk.multinomial(n, occ / occ.sum(), size=100) / occ * fs
    mb = [kaku.code_phase_tuning(cen, y)[1] for y in boot]
    mn = [kaku.code_phase_tuning(cen, y)[1] for y in null]
    p_cf = stats.mannwhitneyu(mb, mn, alternative="greater").pvalue
    rows.append({"mod": mod, "mi": u[f"{mod}_mi_5"], "p_code_curvefit": p_cf, "p_code_grid": u[f"{mod}_p_code_5"]})
cc = pd.DataFrame(rows)
print(f"\nCode scheme on {len(cc)} units with MI > 0.6 ({time.time() - t0:.0f} s): significant at 0.001 by curve_fit {int((cc.p_code_curvefit < .001).sum())}, "
      f"by grid {int((cc.p_code_grid < .001).sum())}, same decision {int(((cc.p_code_curvefit < .001) == (cc.p_code_grid < .001)).sum())}")

In [ ]:
def tuned(mod, order, sig, mi_thr=0.6, p_thr=0.001):
    mi = T[f"{mod}_mi_{order}"]
    if sig is None:
        return (mi > mi_thr).to_numpy()
    return ((mi > mi_thr) & (T[f"{mod}_p_{sig}_{order}"] < p_thr)).to_numpy()
rows = []
for order, oname in ((4, "V2(a) order 4"), (2, "V2(b) order 2"), (5, "code order 5")):
    for sig, sname in (("a", "V3(a)"), ("b", "V3(b)"), ("code", "code 100")):
        L, B = tuned("jaw", order, sig), tuned("br", order, sig)
        rows.append({"filter": oname, "significance": sname, "lick (i)": int(L.sum()), "breath (i)": int(B.sum()), "both": int((L & B).sum()),
                     "lick (ii)": int((L & ~B).sum()), "breath (ii)": int((B & ~L).sum())})
TC = pd.DataFrame(rows)
TC["both/lick"] = TC.both / TC["lick (i)"]; TC["both/breath"] = TC.both / TC["breath (i)"]
def within(x, v, t):
    return abs(x - v) <= t
TC["match (i)"] = within(TC["lick (i)"], 3551, 355) & within(TC["breath (i)"], 1015, 102) & within(TC.both, 182, 18)
TC["match (ii)"] = within(TC["lick (ii)"], 3551, 355) & within(TC["breath (ii)"], 1015, 102) & within(TC.both, 182, 18)
TC["partial"] = (TC["both/lick"] < 0.10) & (TC["both/breath"] < 0.25)
print(TC.round(3).to_string(index=False))
print("\nPaper: licking 3551 (+-355), breathing 1015 (+-102), both 182 (+-18) of 20,300; ratios both/lick 5%, both/breath 18%")
print("Units without a breathing fit (no kept trials or empty bins):", int(T.br_mi_4.isna().sum()), "; jaw:", int(T.jaw_mi_4.isna().sum()))
print("\nBy subject, reading V2(a) V3(b):")
print(pd.DataFrame({"units": T.groupby("subject").size(), "lick": pd.Series(tuned("jaw", 4, "b")).groupby(T.subject.to_numpy()).sum(),
                    "breath": pd.Series(tuned("br", 4, "b")).groupby(T.subject.to_numpy()).sum()}).T.to_string())

**Result: partially reproduced.** Under significance reading V3(a) (the Methods' rank-sum test of
resampled MIs, as the code runs it) the counts match for both filter readings: 3,517 licking, 1,023
breathing and 178 both (order 4), and 3,583, 1,036 and 196 (order 2), against 3,551, 1,015 and 182,
when the licking and breathing counts include the units tuned to both (i). The code's 100-resample
scheme gives the same. Under reading V3(b) (measured MI ranked against random-phase MIs) the counts are
2,759, 602 and 69 (order 4), outside tolerance. The criterion asks for every reading, so the counts do
not hold; both-tuned units are 2.5% to 5.5% of licking units and 11.5% to 18.9% of breathing units under
every reading, which meets the partial rule.

In [ ]:
# Swallow modulation (code get_swallow_tuning, generate_activity_maps.m)
sw_sessions = {r["key"][1] for r in res if len(r["swallow_times"]) > 0}
T["sess_has_sw"] = T.session.isin(sw_sessions)
T["sw_mod_a"] = (T.sw_mi > 0.3) & (T.sw_p <= 0.01) & (T.sw_n >= 20)
T["sw_mod_b"] = (T.sw_mi_b > 0.3) & (T.sw_p <= 0.01) & (T.sw_n >= 20)
rows = []
for v4, col in (("V4(a) larger window rate", "sw_mod_a"), ("V4(b) PSTH peak", "sw_mod_b")):
    S = T[col].to_numpy()
    for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
        L, B = tuned("jaw", order, sig), tuned("br", order, sig)
        for den, dm in (("(a) sessions with a swallow", T.sess_has_sw.to_numpy()), ("(b) >= 20 swallows", (T.sw_n >= 20).to_numpy())):
            for only, om in (("(i) not lick or breath", ~L & ~B), ("(ii) not lick", ~L)):
                rows.append({"V4": v4, "tuning": f"order {order}, {sig}", "denominator": den, "only": only, "units": int(dm.sum()),
                             "swallow only": int((S & om).sum()), "lick and swallow": int((S & L).sum()), "breath and swallow": int((S & B).sum()),
                             "swallow total": int(S.sum())})
SC = pd.DataFrame(rows)
SC["match"] = (abs(SC.units - 18485) <= 1849) & (abs(SC["swallow only"] - 427) <= 43) & (abs(SC["lick and swallow"] - 175) <= 18) & (abs(SC["breath and swallow"] - 55) <= 6)
print(SC.to_string(index=False))
print("\nPaper: 18,485 units; swallow only 427 (+-43), licking and swallowing 175 (+-18), breathing and swallowing 55 (+-6)")
print("Units with >= 20 swallow events:", int((T.sw_n >= 20).sum()), "; sessions with a swallow:", len(sw_sessions), "of", len(res))

**Result: not reproduced.** Reading (a) of the denominator matches: 18,494 units in sessions with a
detected swallow, against 18,485. The swallow-modulated counts do not under any combination: with V4(a)
264 swallow-only (319 under (ii)), 206 licking and swallowing, and 87 breathing and swallowing (order 4,
V3(a)), against 427, 175 and 55; V4(b) gives fewer swallow-only units (187 to 264).

In [ ]:
# Regions the CCF does not label (claims file O3), built from the Allen CCFv3 annotation volume (2017, 25 um;
# kaku.CCF_ANNOTATION_URL), as the verifier asked in round 1. Round 1 built them from the positions of the
# units the files label; both are printed.
vol = kaku.ccf_annotation()
def ids_under(acr):
    i = int(onto.loc[onto.acronym == acr, "id"].iloc[0])
    return onto[onto.structure_id_path.astype(str).str.contains(f"/{i}/")].id.astype(int).tolist()
# check: the annotation at each good unit's CCF position (nearest voxel) against the file's region name
norm = lambda n: " ".join(str(n).replace(",", " ").lower().split())
name_of = dict(zip(onto.id.astype(int), onto.name))
g_ = T.dropna(subset=["ccf_x"])
ix = np.clip((g_[["ccf_z", "ccf_y", "ccf_x"]].to_numpy() / 25).round().astype(int), 0, np.array(vol.shape) - 1)
lab = vol[ix[:, 0], ix[:, 1], ix[:, 2]]
agree = np.mean([norm(name_of.get(int(l), "")) == norm(r) for l, r in zip(lab, g_.region)])
print(f"Annotation volume {vol.shape} (AP, DV, ML); its label at the unit's position equals the file's region name for {agree:.1%} of {len(g_)} good units")
amb = kaku.region_voxels(vol, ids_under("AMB"))
nts = kaku.region_voxels(vol, ids_under("NTS"))
irn = kaku.region_voxels(vol, ids_under("IRN"))
T["ml"] = (T.ccf_x - 5700).abs()
T["apb"] = (T.ccf_z // 100).astype("Int64")
amb_bin = amb.assign(apb=amb.ap // 100).groupby("apb").agg(ml_lo=("ml", "min"), ml_hi=("ml", "max"), dv_lo=("dv", "min"), dv_hi=("dv", "max"))
nts_edge = nts.assign(apb=nts.ap // 100).groupby("apb").ml.max()
print(f"AMB in the annotation: AP {amb.ap.min():.0f} to {amb.ap.max():.0f}, |ML| {amb.ml.min():.0f} to {amb.ml.max():.0f}, DV {amb.dv.min():.0f} to {amb.dv.max():.0f} um "
      f"({len(amb)} voxels, both sides)")
allu = units.dropna(subset=["ccf_x"])
amb_u = allu[allu.acronym.isin(["AMB", "AMBd", "AMBv"])]
print(f"Round 1, from the {len(amb_u)} AMB-labelled units: AP {amb_u.ccf_z.min():.0f} to {amb_u.ccf_z.max():.0f}, |ML| {(amb_u.ccf_x - 5700).abs().min():.0f} to "
      f"{(amb_u.ccf_x - 5700).abs().max():.0f}, DV {amb_u.ccf_y.min():.0f} to {amb_u.ccf_y.max():.0f} um")
irn_ap_mid = (irn.ap.min() + irn.ap.max()) / 2; irn_dv_mid = (irn.dv.min() + irn.dv.max()) / 2
print(f"IRN in the annotation: AP {irn.ap.min():.0f} to {irn.ap.max():.0f} (midpoint {irn_ap_mid:.0f}), DV {irn.dv.min():.0f} to {irn.dv.max():.0f} (midpoint {irn_dv_mid:.0f}) um")
A = amb_bin.reindex(T.apb.astype(float).to_numpy()).set_index(T.index)
in_amb_col = A.ml_lo.notna() & (T.ml >= A.ml_lo) & (T.ml <= A.ml_hi)
in_amb_box = T.ccf_z.between(amb.ap.min(), amb.ap.max()) & T.ml.between(amb.ml.min(), amb.ml.max())
def prebotc(d, step=False):
    """O3 as the claims file words it: within d um ventral of AMB, inside AMB's AP and ML extent (the
    whole nucleus), below its ventral edge. step=True, a variant: AMB's ML extent and ventral edge in the
    unit's own 100 um AP step."""
    if step:
        return (in_amb_col & (T.ccf_y > A.dv_hi) & (T.ccf_y <= A.dv_hi + d)).to_numpy()
    return (in_amb_box & (T.ccf_y > amb.dv.max()) & (T.ccf_y <= amb.dv.max() + d)).to_numpy()
def above_amb(d, step=False):
    if step:
        return (in_amb_col & (T.ccf_y < A.dv_lo) & (T.ccf_y >= A.dv_lo - d)).to_numpy()
    return (in_amb_box & (T.ccf_y < amb.dv.min()) & (T.ccf_y >= amb.dv.min() - d)).to_numpy()
pv_irn = (T.in_IRN & (T.ccf_z > irn_ap_mid) & (T.ccf_y > irn_dv_mid)).to_numpy()
pd_irn_parn = ((T.in_IRN | T.in_PARN) & (T.ccf_z > irn_ap_mid) & (T.ccf_y < irn_dv_mid)).to_numpy()
def lateral_nts(d):
    """Beyond NTS's lateral edge in the same 100 um AP step, within d um."""
    e = pd.Series(nts_edge.reindex(T.apb.astype(float).to_numpy()).to_numpy(), index=T.index)
    return (e.notna() & (T.ml > e) & (T.ml <= e + d) & ~T.in_NTS).to_numpy()
PM = (T.in_P | T.in_MY).to_numpy()
for d in (300, 500):
    print(f"d = {d} um: good units near preBotC {prebotc(d).sum()} (AMB's extent in each AP step: {prebotc(d, True).sum()}), "
          f"above AMB {above_amb(d).sum()} ({above_amb(d, True).sum()}), lateral of NTS {lateral_nts(d).sum()}")
print("posterior-ventral IRN:", pv_irn.sum(), "; posterior-dorsal IRN/PARN:", pd_irn_parn.sum(), "; pons and medulla:", PM.sum())

def enrich(sel, flag, base):
    a = int((sel & flag).sum()); b = int((sel & ~flag).sum())
    c = int((base & flag).sum()); d_ = int((base & ~flag).sum())
    p = stats.fisher_exact([[a, b], [c, d_]], alternative="greater")[1] if a + b > 0 else np.nan
    return {"n": a + b, "tuned": a, "fraction": a / (a + b) if a + b else np.nan, "base fraction": c / (c + d_), "p": p}

rows = []
for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
    B = tuned("br", order, sig)
    for d in (300, 500):
        rows.append({"tuning": f"order {order}, {sig}", "region": f"preBotC ({d} um)", **enrich(prebotc(d), B, PM)})
    for d in (300, 500):
        rows.append({"tuning": f"order {order}, {sig}", "region": f"variant: preBotC by AP step ({d} um)", **enrich(prebotc(d, True), B, PM)})
    rows.append({"tuning": f"order {order}, {sig}", "region": "posterior ventral IRN", **enrich(pv_irn, B, PM)})
BL = pd.DataFrame(rows)
print(BL.round(4).to_string(index=False))
# Mice and sessions behind the preBotC units (verifier, round 2): per subject, good units in the region and those
# breathing-tuned (order 4, V3(a)), whole-nucleus box and AMB's extent per AP step.
B4a = tuned("br", 4, "a")
rows = []
for d in (300, 500):
    for step in (False, True):
        m = prebotc(d, step)
        rows.append(T[m].assign(tuned=B4a[m]).groupby("subject").agg(units=("tuned", "size"), tuned=("tuned", "sum"), sessions=("session", "nunique"))
                    .assign(d=d, extent="by AP step (variant)" if step else "whole nucleus").reset_index())
PBS = pd.concat(rows)
print(PBS.pivot_table(index="subject", columns=["extent", "d"], values=["units", "tuned"], fill_value=0).astype(int).to_string())
print(PBS.groupby(["extent", "d"]).agg(mice=("subject", "nunique"), sessions=("sessions", "sum"), units=("units", "sum"), tuned=("tuned", "sum"),
                                         max_units_one_mouse=("units", "max"), max_tuned_one_mouse=("tuned", "max")).to_string())

**Result: blocked, methods_underspecified.** O3 places the preBotzinger complex within 300 or 500 um
below AMB, inside AMB's AP and ML extent, and does not say whether that extent is the whole nucleus or
the nucleus at each AP level; the two readings give different outcomes, and the claims file's header
blocks such a claim. With the region built from the CCF annotation inside the whole nucleus's AP and
ML extent, 36 and 41 good units lie there, of which 20 or 21 are breathing-tuned (49% to 58%, against
4.4% to 6.7% of pons and medulla units, p < 0.0001). Taking AMB's ML extent and ventral edge in each
100 um AP step leaves 12 or 15 units with 2 or 3 tuned (p 0.04 to 0.27), and 4 of the 10 combinations
of tuning reading and distance fail. The whole-nucleus box takes AMB's deepest point and full ML range
at every AP level, so a unit can count as within 300 um below AMB where it lies further below the local
nucleus; the paper gives no numbers for the region. Posterior ventral IRN holds 88 to 109 of 517 (17% to
21%, p < 0.0001) under every reading, but the criterion needs both regions. The units of the box come
from 10 mice in 12 sessions (at most 10 or 12 from one mouse, DL009), and its 21 tuned units (order 4,
V3(a)) from 7 mice, at most 6 from one; the per-step reading has 12 or 15 units in 5 mice. This was
reproduced, on the whole-nucleus reading, until round 3 of verification, which agreed with the change.

In [ ]:
rows = []
for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
    L = tuned("jaw", order, sig)
    for name, m in (("XII", T.in_XII), ("V", T.in_V), ("IRN", T.in_IRN)):
        rows.append({"tuning": f"order {order}, {sig}", "region": name, **enrich(m.to_numpy(), L, PM)})
    pons = T.in_P.to_numpy()
    rows.append({"tuning": f"order {order}, {sig}", "region": "P (fraction > 1%?)", "n": int(pons.sum()), "tuned": int((pons & L).sum()),
                 "fraction": (pons & L).sum() / pons.sum(), "base fraction": np.nan, "p": np.nan})
LL = pd.DataFrame(rows)
print(LL.round(5).to_string(index=False))

**Result: reproduced.** Licking-tuned units are enriched in XII (36% to 45% of 166 units), V (43%
to 55% of 269) and IRN (34% to 40% of 1,804), against 20% to 26% of pons and medulla units (p < 0.00001
in each, every reading), and make up 15% to 20% of pons units, above 1%.

In [ ]:
base_sw = T.sess_has_sw.to_numpy()
rows = []
for v4, col in (("V4(a)", "sw_mod_a"), ("V4(b)", "sw_mod_b")):
    S = T[col].to_numpy()
    for d in (300, 500):
        for name, m in (("V", T.in_V.to_numpy()), ("IC", T.in_IC.to_numpy()), ("NLL", T.in_NLL.to_numpy()),
                        ("lateral of NTS", lateral_nts(d)), ("above AMB", above_amb(d))):
            rows.append({"V4": v4, "d": d, "region": name, **enrich(m & base_sw, S, base_sw)})
SL = pd.DataFrame(rows)
SL["enriched"] = (SL.fraction > SL["base fraction"]) & (SL.p < 0.05)
print(SL.round(4).to_string(index=False))
print("\nRegions enriched (of 5) per V4 reading and distance:")
print(SL.groupby(["V4", "d"]).enriched.sum().to_string())

**Result: blocked, methods_underspecified.** The two readings of V4 give different outcomes. Under
V4(a), the code's normalisation, swallow-modulated units are enriched in V (5.6%, p = 0.018), IC
(10.9%), NLL (10.1%) and above AMB (8.7% and 7.6%, p = 0.005 and 0.002) against 2.8% in the same
sessions, but not lateral of NTS (1.9% to 2.0%): four of five under both distances, partially
reproduced. Under V4(b), V is not enriched (3.4%, p = 0.095), and above AMB only at 500 um (4.5%, p =
0.037; 4.3%, p = 0.11 at 300 um): two of five at 300 um and three at 500 um, not reproduced. The claims
file's rule blocks a claim whose readings disagree.

In [ ]:
rows = []
for v4, col in (("V4(a)", "sw_mod_a"), ("V4(b)", "sw_mod_b")):
    S = T[col]
    lead = T[S & (T.in_V | T.in_IRN)].sw_peak_t
    follow = T[S & (T.in_IC | T.in_NLL)].sw_peak_t
    mw = stats.mannwhitneyu(lead, follow, alternative="two-sided") if len(lead) and len(follow) else None
    rows.append({"V4": v4, "V and IRN n": len(lead), "V and IRN median peak (s)": lead.median(), "IC and NLL n": len(follow),
                 "IC and NLL median peak (s)": follow.median(), "Mann-Whitney p": mw.pvalue if mw else np.nan})
    for name, m in (("V", T.in_V), ("IRN", T.in_IRN), ("IC", T.in_IC), ("NLL", T.in_NLL)):
        x = T[S & m].sw_peak_t
        print(f"{v4} {name}: n = {len(x)}, median peak {x.median() if len(x) else np.nan:.3f} s, range {x.min() if len(x) else np.nan:.3f} to {x.max() if len(x) else np.nan:.3f}")
print(pd.DataFrame(rows).round(4).to_string(index=False))

**Result: reproduced.** The swallow PSTH of modulated units peaks before swallow onset in V and IRN
(median -1 ms, 111 units, V4(a); -4 ms, 68 units, V4(b)) and after it in IC and NLL (median +16 ms, 53
and 45 units), and the groups differ (Mann-Whitney p < 0.0001 under both readings). The lead in IRN is
small (median -1 ms under V4(a), 98 units); in V it is -11 ms (13 units).

## ed7-glm-separate-clusters

> A GLM on the tracked features found separate clusters of units predicted by licking or by breathing movements.

Blocked, too_large. The cell below measures the cost of the GLM on one session.

In [ ]:
# ed7: cost of the Methods' GLM, measured on one session (verifier, round 1). Reading O8 (a), log link, one
# lag: all good units of the session fitted at once by Newton's method (kaku.fit_poisson_batch).
keys_ = sorted(assets.groupby(["subject", "session"]).groups.keys())
sub_, ses_ = keys_[10]
g_ = assets[(assets.subject == sub_) & (assets.session == ses_)]
t0_ = time.time()
S_ = kaku.read_session(g_.url.iloc[0], dict(zip(g_.probe, g_.url)))
t_read = time.time() - t0_
Xtr, Xte, trm, tem = kaku.glm_design(S_, 0)
Y_ = kaku.glm_counts([u for pr in S_["probes"].values() for u in pr["spikes"]], trm)
t0_ = time.time()
W_ = kaku.fit_poisson_batch(Xtr, Y_)
t_fit = time.time() - t0_
per = t_fit / Y_.shape[1]
n_all = int(units.good.sum())
est_a = per * n_all * 11
print(f"Session {sub_} {ses_}: {Y_.shape[1]} units, {Xtr.shape[0]} training bins of 17 ms, {Xtr.shape[1]} regressors; read in {t_read:.0f} s, "
      f"one lag fitted in {t_fit:.1f} s ({per:.3f} s per unit); weights finite: {bool(np.isfinite(W_).all())}")
print(f"O8 (a), log link, 11 lags, {n_all} units: about {est_a / 3600:.1f} CPU hours; with the logit link as well {2 * est_a / 3600:.1f}; "
      f"O8 (b) adds 5 penalty values per unit and link, each at least one such fit: at least {12 * est_a / 3600:.1f} CPU hours in all, "
      f"against a 60-minute limit for the notebook on 4 CPUs")
del S_, Xtr, Xte, Y_

**Result: blocked, too_large.** The cell above fits one lag of the unpenalised GLM (O8 (a), log link)
for the 176 units of one session and scales the measured time to all 20,109 units and 11 lags: several
CPU hours for O8 (a) with the log link alone, twice that with the logit link as well, and over 100 CPU
hours with the five penalty values of O8 (b) (the printed estimates; the timing depends on the machine's
load), against a 60-minute limit for the notebook. The claim needs every reading of O8, so no part of it
was run.

In [ ]:
def rayleigh(th):
    n = len(th)
    R = np.abs(np.mean(np.exp(1j * th))) * n
    z = R ** 2 / n
    return np.exp(np.sqrt(1 + 4 * n + 4 * (n ** 2 - R ** 2)) - (1 + 2 * n)) if n else np.nan
rows = []
for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
    B = tuned("br", order, sig, mi_thr=0.9)
    for d, step in ((300, False), (500, False), (300, True), (500, True)):
        sel = B & prebotc(d, step)
        ph_ = np.mod(T.loc[sel, f"br_pref_{order}"].to_numpy() - np.pi, 2 * np.pi)   # inspiration onset at 0
        rows.append({"tuning": f"order {order}, {sig}", "d": d, "AMB extent": "by AP step (variant)" if step else "whole nucleus", "units": int(sel.sum()), "inspiration half": int((ph_ < np.pi).sum()),
                     "fraction": np.mean(ph_ < np.pi) if len(ph_) else np.nan, "Rayleigh p": rayleigh(ph_)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
B = tuned("br", 4, "a", mi_thr=0.9)
ph_all = np.mod(T.loc[B, "br_pref_4"].to_numpy() - np.pi, 2 * np.pi)
print(f"All breathing units (MI > 0.9, order 4, V3(a)): {B.sum()}, in the inspiration half {np.mean(ph_all < np.pi):.3f}")
# Mice behind the units of the whole-nucleus box (verifier, round 2), per reading of tuning and distance
rows = []
for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
    B = tuned("br", order, sig, mi_thr=0.9)
    for d in (300, 500):
        sel = B & prebotc(d)
        ph_ = np.mod(T.loc[sel, f"br_pref_{order}"].to_numpy() - np.pi, 2 * np.pi)
        g = pd.DataFrame({"subject": T.loc[sel, "subject"].to_numpy(), "insp": ph_ < np.pi})
        per = g.groupby("subject").insp.agg(["size", "sum"])
        rows.append({"tuning": f"order {order}, {sig}", "d": d, "units": len(g), "mice": len(per), "sessions": T.loc[sel, "session"].nunique(),
                     "per mouse (units, in inspiration half)": "; ".join(f"{s} {n}, {k}" for s, (n, k) in per.iterrows())})
print(pd.DataFrame(rows).to_string(index=False))

**Result: blocked, methods_underspecified, for the reason given under fig2f.** With the whole-nucleus
box, 18 or 19 breathing units with MI > 0.9 lie near the preBotzinger complex under every reading, of
which 14 prefer the inspiration half (74% to 78%), Rayleigh p < 0.0001. With AMB's extent per AP step
only 2 units remain (both in the inspiration half), Rayleigh p 0.15 to 0.17, so the criterion's
p < 0.05 fails. The two readings of the region give different outcomes. Over all 265 breathing units
with MI > 0.9, 40.8% prefer the inspiration half. The 18 or 19 units come from 6 mice in 6 sessions
(DL009 6, DL016 4, DL014 3 and 2 each from DL018, DL023 and DL025); in 4 of the 6 mice most units
prefer the inspiration half, and under every reading DL009 and DL014 hold all the units that do not.
This was reproduced, on the whole-nucleus reading, until round 3 of verification, which agreed with the
change.

In [ ]:
def circ_median(x):
    x = np.asarray(x)
    if not len(x):
        return np.nan
    c = np.linspace(0, 2 * np.pi, 360, endpoint=False)
    return c[np.argmin([np.sum(kaku._circ_dist(x, m)) for m in c])]
prot = {o: {r["key"][1]: circ_median(r["protrusion_phase"][o]) for r in res} for o in kaku.FILTER_ORDERS}
rows = []
for order, sig in ((4, "a"), (4, "b"), (2, "a"), (2, "b"), (5, "code")):
    L = tuned("jaw", order, sig, mi_thr=0.9)
    pm = T.session.map(prot[order]).to_numpy()
    dd = kaku._circ_dist(T[f"jaw_pref_{order}"].to_numpy(), pm)
    rows.append({"tuning": f"order {order}, {sig}", "units": int(L.sum()), "within pi/2 of protrusion": int((L & (dd < np.pi / 2)).sum()),
                 "fraction": np.mean(dd[L] < np.pi / 2)})
print(pd.DataFrame(rows).round(4).to_string(index=False))
pv = pd.Series(prot[4]).dropna()
print("Session circular median protrusion phase (order 4), quartiles (rad):", np.percentile(pv, [25, 50, 75]).round(3))
print("Licks per session with a protrusion frame:", pd.Series({r["key"][1]: len(r["protrusion_phase"][4]) for r in res}).describe()[["min", "50%", "max"]].to_dict())

**Result: reproduced.** 71.4% to 71.5% of licking units with MI > 0.9 (643 to 660 of 899 to 925)
prefer a jaw phase within pi/2 of their session's median phase at full tongue protrusion, under every
reading. The 3D tracks are now read at each session's own trial offset (data check).

## ed8i-ipsilateral-lick-selectivity

> Licking-related neurons showed selectivity for ipsilateral licking, while individual units within each hemisphere had diverse direction preferences, including the center direction.

Blocked, metadata_insufficient; the cell below runs the part that needs no left-right sign.

In [ ]:
# Extended Data Fig 8i, the part that can be run (verifier, round 1). The 3D tracking states no axes or
# sign, so which licks are ipsilateral cannot be fixed (blocked, metadata_insufficient). Without the sign,
# one can still ask whether the lick-side preference differs between the hemispheres (two-sided). Since the
# lateral axis is not named either, both horizontal-looking axes of the 3D tongue track (x, y) are tried.
# Licks are split at the session median of the coordinate, a stand-in for the midline (O9 (b)), or of the
# tip minus the jaw's position before the lick (O9 (a)); per licking unit (O1 (a), order 4, V3 (a)) the rate
# from -50 to 100 ms around lick onset on one side minus the other (kaku.process_session, lick_counts).
lc = {}
for r in res:
    for probe, us in r["units"].items():
        for u in us:
            if "lick_counts" in u:
                lc[(r["key"][1], probe, u["unit_index"])] = u["lick_counts"]
tips = {r["key"][1]: r["lick_tips"] for r in res}
Lu = T[tuned("jaw", 4, "a")]
rows = []
for axis, an in ((0, "x"), (1, "y")):
    for o9 in ("(a) tip minus jaw", "(b) tip"):
        dif, hem = [], []
        for u in Lu.itertuples():
            c = lc.get((u.session, u.probe, u.pos)); lt = tips.get(u.session)
            if c is None or lt is None or not len(lt):
                continue
            x = lt[:, 2 + axis] - (lt[:, 5 + axis] if o9.startswith("(a)") else 0)
            mid = np.nanmedian(x)
            hi, lo = x > mid, x < mid
            if hi.sum() < 10 or lo.sum() < 10:
                continue
            dif.append((c[hi].mean() - c[lo].mean()) / 0.15); hem.append(u.hemi)
        dif, hem = np.array(dif), np.array(hem)
        l_, r_ = dif[hem == "left"], dif[hem == "right"]
        mw = stats.mannwhitneyu(l_, r_, alternative="two-sided") if len(l_) and len(r_) else None
        rows.append({"axis": an, "O9": o9, "units left": len(l_), "units right": len(r_),
                     "median left (spikes/s)": np.median(l_) if len(l_) else np.nan, "median right": np.median(r_) if len(r_) else np.nan,
                     "Mann-Whitney p, two-sided": mw.pvalue if mw else np.nan})
print(pd.DataFrame(rows).round(4).to_string(index=False))
print("Hemisphere from ccf_x against the 5,700 um midline; the files' left/right text is opposite in 99.3% of units (data check).")

**Result: blocked, metadata_insufficient.** Which licks are ipsilateral needs the left-right axis of the
3D tracking and its sign, which the files do not state. The part that needs no sign was run: along the
3D x axis, the lick-side preference of licking units differs between the hemispheres (medians 0.15
against -0.31 spikes/s with the tip against the jaw, 0.22 against -0.26 with the tip alone; 1,714 and
1,803 units; Mann-Whitney p < 0.0001, two-sided), and along the y axis it does not (p 0.30 and 0.81).
So licking units prefer opposite sides in the two hemispheres along x, consistent with a lateral
preference; whether it is ipsilateral cannot be told from the files.

In [ ]:
hi = (T.unit_amp > 500).to_numpy()
print("Good units with amplitude > 500 uV:", int(hi.sum()), "(paper 2223, tolerance 222); of good units:", len(T))
rows = []
for order in (4, 2, 5):
    d = T.assign(hi=hi).groupby("region").agg(n_hi=("hi", "sum"), mi_all=(f"jaw_mi_{order}", "mean"))
    d["mi_hi"] = T[hi].groupby("region")[f"jaw_mi_{order}"].mean()
    d = d[d.n_hi >= 20]
    rho = stats.spearmanr(d.mi_all, d.mi_hi)
    rows.append({"filter order": order, "regions with >= 20": len(d), "Spearman rho": rho.statistic, "p": rho.pvalue})
print(pd.DataFrame(rows).round(5).to_string(index=False))

**Result: reproduced.** 2,366 good units have amplitude above 500 uV (2,223 +- 222), and over the 30
regions with at least 20 of them, their mean licking MI per region correlates with that of all units
(Spearman rho 0.94 to 0.95, p < 0.00001, for each filter order).

In [ ]:
# Figure 4e-f: unit peak differences; O1 unit readings, O2 recording readings, V2 orders 4 and 2.
# Each unit also has the difference for one random split of its cycles (null, verifier round 1), and licking
# units the difference for the code's 40 cycles nearest the median on each side ("code selection").
two_probe = files.groupby(["subject", "session"]).probe.nunique()
two_probe = set(two_probe[two_probe == 2].index.get_level_values(1))
def sim_sessions(Lm, Bm, d):
    """Sessions with two probes, one holding licking units in posterior-dorsal IRN/PARN and the other
    breathing units near the preBotC (O2 (b), O3)."""
    pre = prebotc(d)
    out = {}
    for ses in two_probe:
        s = (T.session == ses).to_numpy()
        lick_p = set(T.loc[s & Lm & pd_irn_parn, "probe"]); br_p = set(T.loc[s & Bm & pre, "probe"])
        pairs = [(a, b) for a in lick_p for b in br_p if a != b]
        if pairs:
            out[ses] = pairs
    return out
O1 = {"(a) MI > 0.6, P < 0.001": dict(mi_thr=0.6), "(b) MI > 0.9, P < 0.001": dict(mi_thr=0.9), "(c) MI > 0.9 alone": dict(mi_thr=0.9, sig=None)}
rows = []
for order in (4, 2):
    for oname, kw in O1.items():
        sig = kw.get("sig", "a")
        for sname in (["a", "b"] if sig is not None else [None]):
            L = tuned("jaw", order, sname, mi_thr=kw["mi_thr"]); B = tuned("br", order, sname, mi_thr=kw["mi_thr"])
            for which, Um, col in (("fig4e licking units", L, "pd_lick_unit"), ("fig4f breathing units", B, "pd_breath_unit")):
                for o2 in ("(a) all sessions", "(b) simultaneous, 300 um", "(b) simultaneous, 500 um"):
                    m = Um.copy()
                    if o2.startswith("(b)"):
                        ss = sim_sessions(L, B, 300 if "300" in o2 else 500)
                        region = pd_irn_parn if which.startswith("fig4e") else prebotc(300 if "300" in o2 else 500)
                        m = m & T.session.isin(ss.keys()).to_numpy() & region
                    x = T.loc[m, col].dropna()
                    nl_ = T.loc[m, col + "_null"]
                    both = T.loc[m, [col, col + "_null"]].dropna()
                    tt = stats.ttest_1samp(x, 0) if len(x) > 2 else None
                    tn = stats.ttest_1samp(nl_.dropna(), 0) if nl_.notna().sum() > 2 else None
                    tp = stats.ttest_rel(both[col], both[col + "_null"]) if len(both) > 2 else None
                    row = {"claim": which, "order": order, "O1": oname, "V3": sname, "O2": o2, "units": len(x),
                           "mean_ms": x.mean() * 1000 if len(x) else np.nan, "p": tt.pvalue if tt else np.nan,
                           "positive": int((x > 0).sum()), "negative": int((x < 0).sum()),
                           "null_mean_ms": nl_.mean() * 1000, "null_p": tn.pvalue if tn else np.nan,
                           "data-null_ms": (both[col] - both[col + "_null"]).mean() * 1000 if len(both) else np.nan,
                           "paired_p": tp.pvalue if tp else np.nan}
                    if col == "pd_lick_unit":
                        xc = T.loc[m, "pd_lick_unit_code"].dropna()
                        row["code_sel_mean_ms"] = xc.mean() * 1000 if len(xc) else np.nan
                        row["code_sel_p"] = stats.ttest_1samp(xc, 0).pvalue if len(xc) > 2 else np.nan
                    rows.append(row)
PDU = pd.DataFrame(rows)
print(PDU.round(4).to_string(index=False))

**Result: both reproduced; fig4f rests on reading O2(a).** Over all sessions (O2(a)), licking units
fire later before a slow breath (mean 61.4 to 68.0 ms; 246 to 705 positive against 26 to 110 negative;
p < 0.001) and breathing units later before a slow lick (33.0 to 33.6 ms; 40 to 61 positive against 1
negative; p < 0.001), under every reading of O1, V2 and V3. Under O2(b), the simultaneous recordings,
3 to 10 licking units give 53.3 to 68.0 ms (p 0.003 to 0.015, every reading), and 2 breathing units
remain, too few for a test. The random-split null (round 1 of verification) separates the two claims.
For breathing units the null is 1.5 to 8.8 ms (p > 0.2) and the data exceed it by 20.0 to 27.1 ms
(paired p < 0.001). For licking units the null is itself large, 47.5 to 54.7 ms (p < 0.0001):
units with MI > 0.6 exceed it by only 3.2 to 5.3 ms (paired p 0.25 to 0.46), units with MI > 0.9 by
18.0 to 19.8 ms (paired p 0.0022 to 0.0059), and under O2(b) not at all. So for licking units most of
the measured difference comes from the slow window starting 60 ms after the fast one. With the 40
cycles nearest the median, as the code selects them, the means over all sessions are 54.8 to 60.7 ms.
The outcomes follow the claims file's criterion, which compares with 0.

In [ ]:
rows = []
for order in (4, 2):
    for oname, kw in O1.items():
        for sname in (["a", "b"] if kw.get("sig", "a") is not None else [None]):
            L = tuned("jaw", order, sname, mi_thr=kw["mi_thr"]); B = tuned("br", order, sname, mi_thr=kw["mi_thr"])
            for d in (300, 500):
                ss = sim_sessions(L, B, d)
                nl, nb = [], []
                for ses, pairs in ss.items():
                    s = (T.session == ses).to_numpy()
                    a, b = pairs[0]
                    nl.append(int((s & L & pd_irn_parn & (T.probe == a).to_numpy()).sum()))
                    nb.append(int((s & B & prebotc(d) & (T.probe == b).to_numpy()).sum()))
                mice = {T.loc[T.session == ses, "subject"].iloc[0] for ses in ss}
                rows.append({"order": order, "O1": oname, "V3": sname, "O3": f"{d} um", "recordings": len(ss), "mice": len(mice),
                             "licking units per recording": f"{min(nl) if nl else 0}-{max(nl) if nl else 0}",
                             "breathing units per recording": f"{min(nb) if nb else 0}-{max(nb) if nb else 0}"})
print(pd.DataFrame(rows).to_string(index=False))
print("Paper: 43 recordings (+-4) in 21 mice (+-2); 1-28 licking and 1-12 breathing units per recording")
print("Two-probe sessions in the dandiset:", len(two_probe), "in", files[files.session.isin(two_probe)].subject.nunique(), "mice")
# Regardless of tuning (verifier, round 1): two-probe sessions with any good unit near the preBotC on one probe
# and any good unit in the posterior-dorsal IRN/PARN on the other; and with AMB's extent taken per AP step
allm = np.ones(len(T), bool)
for d in (300, 500):
    ss = sim_sessions(allm, allm, d)
    mice = {T.loc[T.session == ses, "subject"].iloc[0] for ses in ss}
    print(f"O3 {d} um, any good units: {len(ss)} two-probe sessions in {len(mice)} mice")
_pb = prebotc
for d in (300, 500):
    prebotc = lambda d_, step=False: _pb(d_, True)
    L = tuned("jaw", 4, "a"); B = tuned("br", 4, "a")
    n_t = len(sim_sessions(L, B, d)); n_any = len(sim_sessions(allm, allm, d))
    prebotc = _pb
    print(f"Variant, AMB's extent per AP step, {d} um: {n_t} recordings with O1 (a), order 4, V3 (a); {n_any} with any good units")

**Result: blocked, data_missing.** With the claims file's definitions (O1 and O3, the preBotzinger
complex from the CCF annotation), 2 or 3 two-probe sessions have licking units in posterior-dorsal
IRN/PARN on one probe and breathing units near the preBotzinger complex on the other, in 2 or 3 mice,
against 43 recordings in 21 mice. Regardless of tuning, only 4 two-probe sessions, in 4 mice, have any
good unit in each of the two regions (1 with AMB's extent per AP step), of 162 two-probe sessions in 31
mice. Extended Data Fig 12c analyses the same unit pairs in the baseline of the cued licking task, and
the claims file's header places these recordings in that task, which is not in the dandiset. The
recordings are taken as not deposited; were they among these files, the count would not reproduce.

fig4g-licking-breathing-pairs-correlated, fig4g-symmetric-lags and fig4h-fewer-correlations-outside-oscillator
are blocked for the same reason: their unit pairs are those of these recordings (fig4g, fig4g-symmetric-lags
and fig4h depend on fig4d through fig4g in the claims file), and their cross-correlations were not computed.

In [ ]:
print(f"Notebook run time: {(time.time() - t_start) / 60:.1f} min")

## Summary

| Claim | Outcome |
|---|---|
| code-dataset-link | reproduced |
| units-count | not reproduced |
| region-spike-rates | not reproduced |
| fig1d-lick-breath-frequencies | partially reproduced |
| fig1f-inspiration-between-licks | not reproduced |
| fig1g-lick-delayed-before-slow-breath | partially reproduced |
| fig1f-licking-mode-switch | not reproduced |
| fig1i-breath-delayed-before-slow-lick | reproduced |
| ed1-unidirectional-models-fail | reproduced |
| ed1bc-licks-pause-at-swallows | not reproduced |
| fig1k-rhythms-follow-swallow-at-fixed-latency | reproduced |
| fig1l-lick-interval-preserved | not reproduced |
| fig1l-lick-cycle-prolonged | reproduced |
| fig1n-breath-interval-preserved | reproduced |
| fig1n-breath-cycle-prolonged | reproduced |
| fig2-licking-breathing-tuned-counts | partially reproduced |
| fig2-swallow-modulated-counts | not reproduced |
| fig2f-breathing-units-location | blocked: methods underspecified |
| fig2g-licking-units-location | reproduced |
| fig2h-swallow-units-location | blocked: methods underspecified |
| ed6e-swallow-latencies | reproduced |
| ed8bc-prebotc-inspiration-phase | blocked: methods underspecified |
| ed8e-licking-units-protrusion | reproduced |
| ed4c-licking-map-insensitive-to-sorting | reproduced |
| fig4e-licking-units-anticipate-breath | reproduced |
| fig4f-breathing-units-anticipate-lick | reproduced (assumed) |
| fig4d-simultaneous-recordings, fig4g (two), fig4h | blocked: data missing |
| ed8i-ipsilateral-lick-selectivity | blocked: metadata insufficient |
| ed7-glm-separate-clusters | blocked: too large |
| fig3a, fig3b | blocked: other (public source found, not analysed) |
| ed7g, ed7h, ed12c, ed2, fig3ce, fig3dfh, ed10, fig4b, fig4c, ed11b, fig5b, fig5c, fig5e, fig5f | blocked: data missing |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's. The paper's code (process_mat.py) is ported to the
streamed NWB files and its choices are kept. The tuning curves of the 1000 resamples per unit are fitted
by a grid search where the code uses curve_fit (the measured MI uses curve_fit; on 300 random units the
two agree to a median of 0.0015 in MI, and on 24 tuned units the code's own significance scheme gives
the same decision with either fit). Regions the CCF does not label are built from the CCF annotation
volume as the claims file's O3 words them; fig2f and ed8bc reproduce with AMB's whole extent and do
not with AMB's extent taken per AP step, so both are blocked. For Fig 4e the licking units' 40 fastest and 40 slowest cycles
are used, as the claims file says; the code takes the 40 nearest the median. The peak-difference
measures of Figs 1g and 4e give a positive difference under a random split of the cycles (15.7 ms and
47.5 to 54.7 ms), because the slow window starts later than the fast one; Figs 1i and 4f do not (2.7 ms
and 1.5 to 8.8 ms). Where the claims file names several readings, all are shown above.

What was found about the dataset. The spike index of every file holds start offsets, not end
offsets, so standard NWB readers misassign spikes; 19 files have more histology rows than units; the
behaviour of both probes of a session is the same recording deposited twice; trials carry no session
times; the age field (P2xxW) and the `photostim` stimulus field do not fit the data; the `left`/`right`
location text disagrees with the side `ccf_x` gives; the 3D tracking has no stated axes, and in 19
sessions its trial index is one below the DLC trial number. The cued licking task, the photostimulation
experiments, the rat recordings and the raw videos are not in the dandiset, and it holds 335 insertions
where the paper reports 339. In six subjects (DL029 to DL034) the airflow peaks 20 to 30 ms later against the
tongue than in the others, and fewer inspirations fall inside a protrusion; the files do not say how the
airflow and video clocks were aligned.